# COSC2753 Machine Learning - Assignment 1

**Name:** Kai Nguyen  
**Student ID:** `s4126139`  
**Task:** Regression - predict the continuous target `TARGET_Capacity`  
**Prediction file:** `COSC2753_A1_Predictions_s4126139.csv`

This notebook documents the complete workflow from data understanding and
model evaluation to final model selection and prediction. Development,
model-selection, and final-evaluation stages are separated to reduce data
leakage and support reproducibility.

## Evidence convention

Each major investigation records its objective, method, evidence,
interpretation, and resulting modelling decision.

# PHASE 0 - Notebook Guide, Workflow Map & Reproducibility

## 0.1 Workflow order

Complete Phases 0–3 before model training so that data-quality decisions,
feature roles, and the evaluation strategy are defined independently of model
performance.

All candidate models are compared using the same Country-grouped validation
protocol. The internal holdout is reserved for final confirmation after model
selection, while `test.csv` is used only for schema compatibility checks and
final prediction.

## 0.2 Rubric and evidence map

| Rubric criterion | Main notebook evidence |
|---|---|
| Approach / methodology | Phases 1–5: EDA, evaluation design, preprocessing, baselines, model comparison and tuning |
| Ultimate judgment and analysis | Phase 6: OOF diagnostics, robustness, interpretation, limitations and final model justification |
| Test performance | Country-grouped validation, final internal holdout confirmation, and predictions for `test.csv` |
| Implementation | Reproducible pipelines, grouped splits, validation checks, clear functions and documented code |
| Report and presentation | Phase 8: concise results, figures, decisions and limitations aligned with the final report |

## 0.3 Safety rule - should be checked later

A clean **Restart Kernel → Run All** performs setup, structural audit,
grouped splitting, and lightweight development-only EDA. It does not train
candidate models, inspect holdout targets, inspect external values, or write a
submission file unless the relevant flags are deliberately enabled.


## 0.4 Notebook workflow map

The notebook follows a standard machine-learning workflow adapted to the
requirements of this assignment.

| Phase | Main purpose |
|---|---|
| 1 | Define the prediction task, assignment constraints, evaluation metrics and validation strategy. |
| 2 | Load the supplied files and check schema, column roles, missing values, duplicates and invalid values. |
| 3 | Create the Country-grouped development/holdout split, then perform target-informed EDA using development data only. |
| 4 | Build leakage-safe preprocessing pipelines and establish dummy and linear baselines. |
| 5 | Compare at least three justified regression models using the same grouped folds, then tune the strongest candidates. |
| 6 | Analyse out-of-fold errors, robustness, interpretability and limitations, then justify the final model. |
| 7 | Confirm the selected model on the internal holdout, refit using all labelled data, validate the test schema and export predictions. |
| 8 | Summarise the final evidence, limitations, reproducibility information and submission checks. |

Schema and data-quality checks are completed before splitting because they do
not use target relationships to select a model. Target-informed EDA is
performed only after the internal holdout has been created and uses the
development partition only.

## 0.5 Imports and environment

In [1]:
# Standard library
from pathlib import Path
import hashlib
import json
import platform
import time
import warnings

# Data and visualisation
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

# Scikit-learn
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    GroupShuffleSplit,
    cross_val_predict,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    PolynomialFeatures,
    StandardScaler,
)

# Display formatting only; underlying values are unchanged.
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
warnings.filterwarnings("default")

print(f"Python:       {platform.python_version()}")
print(f"pandas:       {pd.__version__}")
print(f"NumPy:        {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"Matplotlib:   {matplotlib.__version__}")

Python:       3.14.5
pandas:       3.0.3
NumPy:        2.5.1
scikit-learn: 1.9.0
Matplotlib:   3.11.0


## 0.6 Core configuration

In [2]:
STUDENT_ID = "s4126139"

# Reproducibility and compute settings
RANDOM_STATE = 42
N_JOBS = -1  # Use all available CPU cores for supported operations
np.random.seed(RANDOM_STATE)

# Predeclared evaluation settings:
# - Five Country-grouped folds are used for model selection.
# - 20% of unique Country groups are reserved for one-time final evaluation.
# - Group and row balance are verified after splitting.
N_GROUP_FOLDS = 5
INTERNAL_HOLDOUT_FRACTION = 0.20

# Column roles
ID_COL = "RecordID"             # Output mapping only; not a predictor
GROUP_COL = "Country"           # Grouping unit for holdout and CV
TARGET_COL = "TARGET_Capacity"  # Continuous regression target
CASE_COUNT_COL = "CaseCount"  # Canonical name; test uses "Case count"

## 0.7 Decision state and execution controls

In [3]:
# =============================================================================
# Decision state
# Values are completed only after the corresponding phase provides evidence.
# =============================================================================

# PHASE 1 — Problem definition and evaluation contract
# Select one primary metric after documenting its rationale.
PRIMARY_METRIC_KEY = "mae"  # "rmse", "mae", or "r2"

# PHASES 2–4 — Data quality, EDA and feature policy
# Set to True only after all development-data policies below are resolved.
DEVELOPMENT_POLICY_CONFIRMED = False

# PHASE 2 — Data-quality policy
# Controls how metadata-invalid Status values are handled.
STATUS_INVALID_POLICY = None  # "retain_numeric" or "invalid_to_nan"

# PHASE 4 — Country feature policy
# Country is always the grouping unit; this controls whether it is also
# included as an encoded model feature.
COUNTRY_FEATURE_POLICY = None  # "group_only" or "one_hot"

# PHASES 3–4 — Optional evidence-based feature exclusions
# Mandatory non-features such as RecordID, Country and target are handled
# separately by the column-role logic. Additional exclusions require evidence.
ADDITIONAL_FEATURE_EXCLUSIONS = []


# =============================================================================
# Execution controls
# Enable a stage only after the preceding decision requirements are complete.
# =============================================================================

# PHASE 4 — Preprocessing and baseline development
RUN_PREPROCESSING_SMOKE_TEST = False
RUN_POLICY_ABLATIONS = False
RUN_BASELINE_EVALUATION = False

# PHASE 5 — Candidate comparison and hyperparameter tuning
RUN_MODEL_COMPARISON = False
RUN_TUNING = False

# PHASE 6 — Error analysis, robustness and final model selection
RUN_OOF_ANALYSIS = False
RUN_ROBUSTNESS_TESTS = False
RUN_FINALIST_ABLATIONS = False
RUN_MODEL_INTERPRETATION = False

# PHASE 7 — Final evaluation, compatibility check and prediction
RUN_FINAL_HOLDOUT = False
RUN_EXTERNAL_COMPATIBILITY_AUDIT = False
RUN_FINAL_SUBMISSION = False


# =============================================================================
# Model-selection state
# Populated from evidence produced in Phases 5–6.
# =============================================================================

# PHASE 5 — Models retained after grouped-CV comparison and tuning
FINALIST_SPECS = {}

# PHASE 6 — Finalist selected for detailed out-of-fold diagnostics
OOF_DIAGNOSTIC_LABEL = None

# PHASE 6 — Final model specification
# Set to True only after the model, parameters, feature set and data policies
# have been justified using development evidence.
FINAL_SPEC_CONFIRMED = False

FINAL_MODEL_SPEC = {
    "candidate": None,
    "params": {},
    "features": None,
    "status_invalid_policy": None,
    "country_feature_policy": None,
}

# PHASE 6 — Grouped-CV primary score used as the holdout reference
FINAL_CV_PRIMARY_REFERENCE = None


# =============================================================================
# Final-evaluation rules
# Complete in Phase 6 before enabling Phase 7 holdout access.
# =============================================================================

# PHASE 6 — Confirms that the holdout acceptance rule is predeclared
HOLDOUT_ACCEPTANCE_RULE_CONFIRMED = False

# PHASE 6 — Maximum acceptable degradation from grouped CV to holdout
HOLDOUT_MAX_RELATIVE_PRIMARY_DEGRADATION = None

# PHASE 6 — Maximum acceptable absolute mean prediction bias
HOLDOUT_MAX_ABSOLUTE_BIAS = None

# PHASE 7 — Set to True after recording the holdout result and decision
HOLDOUT_DECISION_RECORDED = False

## 0.8 Portable paths and input files - CONFIG AGAIN BEFORE SUBMIT

In [4]:
def locate_assignment_dir() -> Path:
    """Locate ASM1 from the supported Jupyter launch locations."""
    candidates = [
        Path.cwd(),
        Path.cwd() / "Machine Learning at RMIT" / "ASM1",
    ]

    for candidate in candidates:
        marker = (
            candidate
            / "2026B_dataset"
            / "2026A_dataset"
            / "train.csv"
        )
        if marker.exists():
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not locate ASM1. Start Jupyter from either the ASM1 folder "
        "or the Machine_Learning workspace root."
    )


ASSIGNMENT_DIR = locate_assignment_dir()
DATA_DIR = ASSIGNMENT_DIR / "2026B_dataset" / "2026A_dataset"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_PATH = DATA_DIR / "s1234567_predictions.csv"

required_paths = {
    "training data": TRAIN_PATH,
    "test data": TEST_PATH,
    "prediction template": SAMPLE_PATH,
}

missing_paths = {
    label: path
    for label, path in required_paths.items()
    if not path.exists()
}

if missing_paths:
    details = ", ".join(
        f"{label}: {path}"
        for label, path in missing_paths.items()
    )
    raise FileNotFoundError(
        f"Missing required assignment file(s): {details}"
    )

print(f"Assignment directory: {ASSIGNMENT_DIR}")
print(f"Data directory:       {DATA_DIR}")

Assignment directory: C:\Users\Khoai\RMIT\Machine_Learning\Machine Learning at RMIT\ASM1
Data directory:       C:\Users\Khoai\RMIT\Machine_Learning\Machine Learning at RMIT\ASM1\2026B_dataset\2026A_dataset


# PHASE 1 — Problem Definition & Evaluation Contract

**Phase objective.** Define the regression problem and freeze the evaluation
rules before any model comparison. This phase specifies the prediction target,
assignment constraints, dataset roles, intended generalisation setting,
performance metrics, validation design, and minimum evidence required for
model selection.

| Section | Evaluation contract item |
|---|---|
| 1.1 | Define the task, experience, performance, and prediction unit |
| 1.2 | Record assignment constraints and modelling non-goals |
| 1.3 | Separate the roles of development data, the locked internal holdout, and external test data |
| 1.4 | State and justify the Country-level generalisation setting |
| 1.5 | Define the primary and secondary metrics and their optimisation directions |
| 1.6 | Freeze five-fold Country-grouped cross-validation and the 20% Country-group holdout |
| 1.7 | Predeclare baselines and success criteria |

**Workflow safeguards.** Model selection must not use the locked internal
holdout or `test.csv`. The holdout is reserved for one final confirmation after
development, while `test.csv` is used only for compatibility checks and final
prediction. Later training, holdout evaluation, and export steps remain
controlled by explicit execution flags.

## 1.1 Task–Experience–Performance

| Element | Assignment-specific definition |
|---|---|
| **Task** | Estimate the continuous `TARGET_Capacity` of a chatbot/LLM system from eligible technological and operational attributes available at prediction time. |
| **Prediction unit** | One dataset row representing the technological and operational profile of one chatbot/LLM-system observation; the required output is one continuous capacity estimate per row. |
| **Experience** | Valid labelled observations in `train.csv`. `Country` is retained as the grouping variable for development resampling and construction of the locked internal holdout. |
| **Performance** | Generalisation performance is estimated using predeclared metrics under Country-grouped cross-validation, followed by one final confirmation on the locked Country-group holdout after model selection. |

**Mathematical formulation.** For observation $i$, let $y_i$ denote the observed
`TARGET_Capacity`. The prediction contract is

$$
\widehat{y}_i = h_{\theta}(\mathbf{x}_i).
$$

Here, $\mathbf{x}_i$ contains only predictors that are eligible and available at external
prediction time, while $\theta$ denotes the parameters learned from
the permitted training data. `RecordID` is excluded from the feature vector; whether `CaseCount` and
`Country` are included as predictors remains an evidence-based decision for later phases.

## 1.2 Assignment Constraints and Modelling Boundaries

- **Model coverage:** Report results for at least three distinct regression
  approaches. At least two must use techniques covered through Week 5; an
  additional beyond-class approach is encouraged but not required.
- **Identifier exclusion:** `RecordID` is a row identifier and must never be
  used as a predictor.
- **Schema compatibility:** The raw external feature `Case count` is
  deterministically renamed to the metadata-defined `CaseCount`, making it
  available in both training and external prediction data. Its use as a
  predictor remains subject to proxy-risk analysis, ablation evidence, and
  documented justification.
- **Evidence-based feature removal:** Any predictor exclusion beyond the
  mandatory non-features must be supported by data evidence and documented
  justification.
- **Implementation evidence:** The submitted notebook/code must be
  understandable, runnable, and contain evidence for every investigation and
  result discussed in the report or presentation.
- **Deliverable alignment:** The PDF report, presentation, notebook/code and
  prediction CSV must describe the same modelling workflow. The CSV must be
  generated by the frozen ultimate model, preserve the required schema and row
  order, and be named `COSC2753_A1_Predictions_s4126139.csv`.

**Modelling boundary.** These constraints do not preselect the final algorithm,
additional feature exclusions, or whether `Country` is used as a predictor.
Those decisions remain evidence-based and are owned by later phases.

## 1.3 Dataset Roles and Permitted Use

| Data object | Role and permitted use |
|---|---|
| **Labelled source — `train.csv`** | Raw source for supervised development. Only rows with an observed `TARGET_Capacity` are eligible for the modelling pool; Phase 2 implements and verifies their exclusion without target imputation. The eligible rows are partitioned by `Country` into development and locked holdout sets. |
| **Development partition** | Used for EDA, data-quality and feature-policy decisions, Country-grouped cross-validation, model comparison, hyperparameter tuning, and finalist selection. |
| **Locked internal holdout** | Contains 20% of the labelled Country groups and is reserved for one final confirmation after the complete model specification is frozen. It must not influence EDA, preprocessing, feature selection, tuning, or finalist selection. |
| **External test set — `test.csv`** | Contains predictors but no `TARGET_Capacity`. It is used only for schema and I/O compatibility checks and for final prediction with the frozen pipeline. External feature values must not guide feature, model, or hyperparameter selection. `RecordID` and the original row order are preserved for submission. |

**Schema normalisation.** The raw external file names the metadata-defined
`CaseCount` feature as `Case count`. A documented deterministic rename aligns
the predictor schemas; after normalisation, `TARGET_Capacity` is the only
train-only column.

## 1.4 Country-level Generalisation Scenario

The primary offline protocol is designed to estimate generalisation to held-out
`Country` groups rather than to randomly held-out rows. `Country` is therefore
predeclared as the grouping unit for cross-validation and the locked internal
holdout, preventing observations from the same Country from appearing in both
fitting and evaluation partitions.

This is an evaluation hypothesis rather than a deployment claim. Phase 2
examines whether the observed Country structure supports this design, and
Phase 3 verifies the resulting group separation. Whether `Country` is also used
as a model predictor remains a separate, evidence-based decision.


## 1.5 Evaluation Metric Contract

The supplied assignment materials do not specify the hidden-test scoring metric
or a domain cost function that assigns disproportionate importance to large
errors. MAE is therefore predeclared as the primary offline model-selection
metric because it is expressed in the target units, is directly interpretable
as the typical absolute prediction error, and makes the minimal assumption that
prediction cost increases linearly with error magnitude.

| Metric | Role | Direction | Interpretation |
|---|---|---|---|
| **MAE** | Primary selection metric | Minimise | Measures the typical absolute error in `TARGET_Capacity` units without allowing a small number of extreme errors to dominate the comparison. |
| **RMSE** | Secondary large-error diagnostic | Minimise | Places greater emphasis on large prediction errors and reveals whether a model occasionally produces severe misses. |
| **R²** | Secondary relative diagnostic | Maximise | Measures improvement relative to a mean-based reference, but is not used as the sole model-ranking criterion. |

Models are ranked by mean validation MAE under the common evaluation protocol
defined in Section 1.6. Per-fold metric values and mean ± standard deviation are
reported as descriptive evidence of stability across validation partitions;
the fold standard deviation is not interpreted as a formal confidence interval.

RMSE and R² are supporting diagnostics rather than alternative criteria selected
after viewing the results. If MAE and RMSE produce materially different model
rankings, the primary metric is not changed retrospectively. The disagreement
is reported and investigated through fold stability, residual analysis, and
large-error diagnostics before the ultimate model judgment.

In [16]:
SCORING = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
}
LOSS_METRICS = {"rmse", "mae"}

def validate_primary_metric() -> str:
    assert PRIMARY_METRIC_KEY in SCORING, (
        f"PRIMARY_METRIC_KEY must be one of {sorted(SCORING)}; "
        f"got {PRIMARY_METRIC_KEY!r}."
    )
    return PRIMARY_METRIC_KEY

def scorer_value_for_display(raw_score: float, metric_key: str) -> float:
    return -raw_score if metric_key in LOSS_METRICS else raw_score

metric_registry = pd.DataFrame(
    [
        ["mae", "minimise", "primary: typical absolute error in target units"],
        ["rmse", "minimise", "secondary: sensitivity to large errors"],
        ["r2", "maximise", "secondary: improvement over a mean-based reference"],
    ],
    columns=["metric", "direction", "role"],
)

with pd.option_context("display.max_colwidth", None):
    display(metric_registry)


,metric,direction,role
0,mae,minimise,primary: typical absolute error in target units
1,rmse,minimise,secondary: sensitivity to large errors
2,r2,maximise,secondary: improvement over a mean-based reference


## 1.6 Evaluation Framework Contract

The following evaluation protocol is frozen before model comparison.

| Component | Frozen decision |
|---|---|
| Prediction unit | One row representing a unique Country–Year observation |
| Grouping and generalisation unit | `Country`; no Country group may cross fitting and evaluation partitions |
| Internal holdout | `GroupShuffleSplit` reserves 20% of labelled Country groups using `random_state=42` |
| Holdout role | One-time final confirmation after the complete model specification has been frozen |
| Development validation | Shuffled five-fold `GroupKFold` with `random_state=42` |
| Comparison fairness | The same Country-group fold assignments are reused for all model candidates and tuning procedures |
| Performance measures | MAE is the primary selection metric; RMSE and R² are supporting diagnostics |
| External test data | Used only for schema/I/O compatibility during development and final inference after model freeze; it does not guide feature or model selection |

The 20% holdout allocation refers to the proportion of Country groups rather
than an exact proportion of rows. The realised group and row proportions,
split integrity, and partition identifiers are verified and recorded in
Phase 3.

In [17]:
primary_metric = validate_primary_metric()
secondary_metrics = [
    metric_key
    for metric_key in SCORING
    if metric_key != primary_metric
]

evaluation_contract = {
    "prediction_unit": "one row (unique Country-Year observation)",
    "unit_of_generalisation": GROUP_COL,
    "group_separation_rule": (
        "no Country group may cross fitting and evaluation partitions"
    ),
    "holdout_splitter": "GroupShuffleSplit",
    "internal_holdout_group_fraction": INTERNAL_HOLDOUT_FRACTION,
    "holdout_role": (
        "one-time confirmation after complete model-specification freeze"
    ),
    "group_cv_splitter": "GroupKFold(shuffle=True)",
    "group_cv_folds": N_GROUP_FOLDS,
    "fold_reuse": "identical Country-group folds for all candidates",
    "primary_metric": primary_metric,
    "secondary_metrics": secondary_metrics,
    "external_test_role": (
        "schema/I/O compatibility during development; "
        "final inference after model freeze"
    ),
    "random_state": RANDOM_STATE,
}

with pd.option_context("display.max_colwidth", None):
    display(
        pd.Series(
            evaluation_contract,
            name="Evaluation contract",
        )
    )

prediction_unit                                                          one row (unique Country-Year observation)
unit_of_generalisation                                                                                     Country
group_separation_rule                                 no Country group may cross fitting and evaluation partitions
holdout_splitter                                                                                 GroupShuffleSplit
internal_holdout_group_fraction                                                                             0.2000
holdout_role                                       one-time confirmation after complete model-specification freeze
group_cv_splitter                                                                         GroupKFold(shuffle=True)
group_cv_folds                                                                                                   5
fold_reuse                                                        identical Coun

## 1.7 Baselines and Model-Selection Success Criteria

All reference models and candidates are evaluated using the same
Country-grouped folds and metric definitions.

| Reference model | Purpose |
|---|---|
| `DummyMean` | Establishes the minimum benchmark: a useful model must outperform prediction by the training-fold target mean. |
| `CaseCountOffset` | Provides a problem-specific diagnostic benchmark using `CaseCount` plus an offset learned within each training fold. It is considered deployment-eligible only if later evidence supports the legitimacy of `CaseCount`. |
| `LinearRegression_W2` | Provides a transparent learned baseline against which additional model complexity can be justified. |

A model demonstrates basic predictive value only if it improves on
`DummyMean` in mean Country-grouped validation MAE. The practical comparison
is then made against the strongest eligible transparent baseline. Additional
complexity is justified only when its MAE improvement is not offset by
substantially poorer fold stability, a larger train–validation gap, severe
RMSE errors, unreasonable runtime, or weaker interpretability.

No arbitrary percentage-improvement threshold is imposed because the supplied
materials do not define a domain cost function or practical error tolerance.
When models have similar mean MAE, preference is given to the model with more
stable grouped-fold performance, a smaller generalisation gap, stronger
supporting diagnostics, and fewer robustness or deployment concerns. If no
complex candidate convincingly improves on the strongest eligible baseline,
the simpler baseline remains the preferred recommendation and this result is
reported transparently.

These baselines establish reference performance; they do not replace the
assignment requirement to compare and report at least three eligible
regression models.

# PHASE 2 — Data Loading, Structural Audit & Data Quality

## Purpose and scope

This phase establishes a reproducible and assignment-compliant data foundation
before any target-informed EDA or modelling. It loads the required files,
verifies the schema, identifier and submission contracts, applies only
documented deterministic schema normalisation, assigns structural column roles,
and records missingness, key violations and metadata inconsistencies.

The row with a missing `TARGET_Capacity` is retained in the raw audit data but
excluded from the supervised modelling pool because it has no observed label.
The target is never imputed or fabricated.

## Data-use and leakage boundary

- **Labelled source — `train.csv`:** Structural facts may be audited, including
  dimensions, columns, dtypes, identifiers, keys, missingness and
  metadata-validity anomalies. This phase does not use predictor–target
  relationships to select features, models or hyperparameters.
- **External sources — `test.csv` and the sample submission:** These files are
  used only to verify schema, row/identifier order, the deterministic
  `Case count` → `CaseCount` rename, and I/O compatibility. External feature
  distributions, categories, ranges and missingness do not inform development
  decisions.
- **Locked internal holdout:** The Country-grouped holdout is created in
  Phase 3 before target-informed EDA. Once created, it remains untouched until
  the final confirmation phase.
- **Out-of-scope operations:** Model fitting, feature selection, hyperparameter
  tuning, final inference and prediction-file export are not performed in this
  phase.

## Decision ownership

Phase 2 distinguishes among **observed structural facts**, **anomalies**,
**candidate actions** and **final policies**. Missing-target exclusion and the
non-silent handling contract for metadata-invalid `Status` are owned by the
data-quality process. Decisions about using `Country` as an encoded feature,
the predictive legitimacy of `CaseCount`, and any additional feature
exclusions remain unresolved until their designated development phases provide
the required evidence.

| Notebook steps | Phase 2 responsibility |
|---|---|
| 2.1–2.3 | Inventory and load the sources; verify dimensions, submission compatibility and normalised schemas. |
| 2.4 | Assign structural column roles and enforce mandatory non-feature roles such as `RecordID` and the target. |
| 2.5–2.9 | Detect and document missingness, duplicate/key issues, dtype or range anomalies, Country–Year structure and repeatable validation checks. |
| 2.10 | Record Phase-2-owned decisions, route later-phase decisions to their owners, and evaluate the Phase 2 completion gate. |

Phase 2 is complete only when its structural assertions pass, the supervised
modelling pool is reproducibly defined, Phase-2-owned rules are documented,
later-phase decisions remain explicitly unresolved, and neither external-test
values nor locked-holdout evidence has influenced development.

## 2.1 Source inventory and raw loading

The required training, external-test and prediction-template CSV files are
validated and fingerprinted with SHA-256 to document the exact input snapshot
used by this notebook. Only file-level metadata is reported here. The external
test file is loaded for later schema and I/O compatibility checks, but its
feature values are not summarised or used to make development decisions.

In [18]:
# ============================================================================
# NOTEBOOK STEP 2.1 — Source inventory, SHA-256 fingerprints and raw loading
#
# Data scope:
# - Validate and fingerprint the three required raw CSV sources.
# - Load test.csv for later structural compatibility checks only.
# - Do not summarise or compare external feature values in this step.
#
# Expected evidence:
# - Every required source is a file.
# - Relative paths, file sizes and complete SHA-256 fingerprints are recorded.
# - All three raw CSV files load without parser errors.
# ============================================================================

# Store one audit record for each required source file. These records will be
# converted to a DataFrame so that the source checks remain visible in the
# executed notebook.
source_inventory_records = []

for source_name, path in required_paths.items():
    # is_file() verifies both existence and file type. A directory at the
    # expected path would therefore fail this check.
    is_file = path.is_file()

    source_inventory_records.append(
        {
            # Human-readable role of the source, such as train, test or sample.
            "source": source_name,

            # Record a portable path relative to the assignment directory
            # rather than a machine-specific absolute path.
            "relative_path": path.relative_to(
                ASSIGNMENT_DIR
            ).as_posix(),

            # Preserve the validation result in the displayed audit table.
            "is_file": is_file,

            # Record the file size as an additional reproducibility check.
            # None is used when the file is unavailable so that stat() is not
            # called on a missing path.
            "bytes": path.stat().st_size if is_file else None,

            # A SHA-256 fingerprint identifies the exact raw file contents.
            # The complete hexadecimal digest is retained so that any later
            # change to a source file can be detected.
            "sha256": (
                hashlib.sha256(path.read_bytes()).hexdigest()
                if is_file
                else None
            ),
        }
    )

# Convert the list of audit records into a tabular source inventory.
source_inventory = pd.DataFrame(source_inventory_records)

# Collect the relative paths of any required sources that are missing or are
# not regular files.
missing_sources = source_inventory.loc[
    ~source_inventory["is_file"],
    "relative_path",
].tolist()

# Stop execution before attempting CSV loading if any required source failed
# validation. The error reports all missing paths in one message.
if missing_sources:
    raise FileNotFoundError(
        "Missing required source file(s): "
        + ", ".join(missing_sources)
    )

# Display full SHA-256 fingerprints without permanently changing pandas'
# global display configuration.
with pd.option_context("display.max_colwidth", None):
    display(source_inventory)

# Load immutable raw-source snapshots. No cleaning, schema normalisation,
# target filtering or predictor transformation is performed in this step.
train_source = pd.read_csv(TRAIN_PATH)
test_source = pd.read_csv(TEST_PATH)
template_source = pd.read_csv(SAMPLE_PATH)

# Reaching this statement confirms that all three CSV files were found and
# parsed without raising a file or CSV parser error.
print("Raw CSV sources loaded successfully.")

,source,relative_path,is_file,bytes,sha256
0,training data,2026B_dataset/2026A_dataset/train.csv,True,267548,f38cdd692aadbccfce5fb605de0c5565755201e14b82e9a43918dc561031260c
1,test data,2026B_dataset/2026A_dataset/test.csv,True,107082,afd81ad143224785d51b73fb3e8d3a3ed06e7e4b999b46c3d367ab7008e291b1
2,prediction template,2026B_dataset/2026A_dataset/s1234567_predictions.csv,True,5979,84f16dc7d74757c2e4d74902ebabd5bc2ac5976d76b42404bdf293f84245e0e4


Raw CSV sources loaded successfully.


## 2.2 Dataset dimensions and prediction-template schema

This step records the dimensions of the three raw data objects and verifies
that the supplied prediction template has the required ordered columns and one
row per external-test observation. Predictor-schema compatibility is evaluated
in Section 2.3, while exact identifier and row-order integrity is evaluated in
Section 2.6.

In [19]:
# ============================================================================
# NOTEBOOK STEP 2.2 — Dataset dimensions and prediction-template schema
#
# Scope: Record raw dimensions and verify the template's column order and row
# count. Predictor-schema and exact identifier-order checks remain in Steps
# 2.3 and 2.6 respectively. External feature values are not inspected here.
#
# Expected evidence: dimensions for all three objects and a passing template
# contract with columns ["ID", TARGET_Capacity] and one row per test record.
# ============================================================================

# Build a compact structural summary without inspecting feature values.
raw_objects = {
    "train.csv": train_source,
    "test.csv": test_source,
    "prediction template": template_source,
}

shape_summary = pd.DataFrame(
    {
        name: {"rows": len(data), "columns": data.shape[1]}
        for name, data in raw_objects.items()
    }
).T.astype(int)

display(shape_summary)

# The prediction template must preserve the required two-column order and
# contain exactly one row for every external-test observation.
expected_template_columns = ["ID", TARGET_COL]
template_contract = pd.Series(
    {
        "columns_match_required_order": template_source.columns.tolist()
        == expected_template_columns,
        "row_count_matches_test": len(template_source) == len(test_source),
    },
    name="passes",
)

display(template_contract.to_frame())

# Stop early with observed and expected details if either condition fails.
if not template_contract.all():
    raise ValueError(
        "Prediction-template contract failed. "
        f"Observed columns={template_source.columns.tolist()}, "
        f"expected columns={expected_template_columns}, "
        f"template rows={len(template_source)}, test rows={len(test_source)}."
    )

,rows,columns
train.csv,2071,26
test.csv,867,25
prediction template,867,2


,passes
columns_match_required_order,True
row_count_matches_test,True


## 2.3 Deterministic schema normalisation and compatibility

The supplied training data uses the metadata-defined name `CaseCount`, whereas
the external file uses `Case count`. A single deterministic rename standardises
this known discrepancy without using target information or external feature
distributions. The raw source objects remain unchanged.

After normalisation, `TARGET_Capacity` must be the only train-only column and
all shared columns must have compatible dtype families. Exact integer/float
representation differences are reported but are not treated as semantic
numeric-type conflicts. No imputation, value correction or feature-selection
decision is performed in this step.

In [20]:
# ============================================================================
# STEP 2.3 — Deterministic schema normalisation and compatibility
# ============================================================================

EXTERNAL_CASE_COUNT_ALIAS = "Case count"

# Verify the documented raw schemas before renaming.
assert CASE_COUNT_COL in train_source.columns
assert EXTERNAL_CASE_COUNT_ALIAS not in train_source.columns
assert EXTERNAL_CASE_COUNT_ALIAS in test_source.columns
assert CASE_COUNT_COL not in test_source.columns

# Preserve the source DataFrames and normalise only the known external alias.
train_normalised = train_source.copy()
test_normalised = test_source.rename(columns={EXTERNAL_CASE_COUNT_ALIAS: CASE_COUNT_COL}).copy()

print("Column rename: test.csv 'Case count' -> 'CaseCount'")


# Compare the normalised column sets.
train_only = train_normalised.columns.difference(test_normalised.columns).tolist()
test_only = test_normalised.columns.difference(train_normalised.columns).tolist()

schema_summary = pd.DataFrame(
    {
        "Observed": [
            ", ".join(train_only) or "None",
            ", ".join(test_only) or "None",
        ],
        "Expected": [TARGET_COL, "None"],
        "Status": [
            "PASS" if train_only == [TARGET_COL] else "FAIL",
            "PASS" if not test_only else "FAIL",
        ],
    },
    index=["Columns only in train", "Columns only in test"],
)

display(schema_summary)


# Compare pandas dtypes for every shared column.
shared_columns = train_normalised.columns.intersection(test_normalised.columns)

dtype_comparison = pd.concat(
    [
        train_normalised[shared_columns].dtypes.rename("Train dtype"),
        test_normalised[shared_columns].dtypes.rename("Test dtype"),
    ],
    axis=1,
)

dtype_differences = dtype_comparison.loc[
    dtype_comparison["Train dtype"]
    != dtype_comparison["Test dtype"]
].copy()

numeric_in_both = (
    train_normalised.select_dtypes(include="number")
    .columns.intersection(
        test_normalised.select_dtypes(include="number").columns
    )
)

dtype_differences["Status"] = np.where(
    dtype_differences.index.isin(numeric_in_both),
    "Compatible numeric",
    "Review required",
)

display(dtype_differences)


# Enforce the structural contract.
assert train_normalised.columns.is_unique
assert test_normalised.columns.is_unique
assert train_only == [TARGET_COL]
assert not test_only
assert ID_COL in shared_columns
assert TARGET_COL not in test_normalised.columns
assert pd.api.types.is_numeric_dtype(train_normalised[TARGET_COL])
assert dtype_differences["Status"].eq("Compatible numeric").all()

print(
    f"Schema contract: PASS. {len(shared_columns)} shared columns checked; "
    f"{len(dtype_differences)} exact dtype differences are numeric-compatible."
)

Column rename: test.csv 'Case count' -> 'CaseCount'


,Observed,Expected,Status
Columns only in train,TARGET_Capacity,TARGET_Capacity,PASS
Columns only in test,None,None,PASS


,Train dtype,Test dtype,Status
CaseCount,float64,int64,Compatible numeric
Year,float64,int64,Compatible numeric
ModelFailureRate-NT,float64,int64,Compatible numeric
AILearningStability,int64,float64,Compatible numeric
PatchCoverageRate,float64,int64,Compatible numeric


Schema contract: PASS. 25 shared columns checked; 5 exact dtype differences are numeric-compatible.


## 2.4 Column-role assignment

Each normalised training column is assigned one structural role before
missingness analysis or modelling. `RecordID` is retained only for output
mapping, `TARGET_Capacity` is the regression target, and `Country` is the
grouping variable. All remaining columns are provisional candidate predictors.

This assignment does not approve final feature use. `CaseCount` remains subject
to proxy-risk evidence and ablation, while the encoded use of `Country` is
decided separately in Phase 4.

In [21]:
# ============================================================================
# STEP 2.4 — Column-role assignment
# ============================================================================

required_role_columns = {
    ID_COL,
    GROUP_COL,
    TARGET_COL,
    CASE_COUNT_COL,
}
assert required_role_columns.issubset(train_normalised.columns)

# Start with the default role, then override the special columns.
column_roles = pd.Series(
    "candidate predictor",
    index=train_normalised.columns,
    name="Role",
)

column_roles.loc[ID_COL] = "identifier / output mapping only"
column_roles.loc[GROUP_COL] = "grouping variable; feature policy unresolved"
column_roles.loc[TARGET_COL] = "regression target"
column_roles.loc[CASE_COUNT_COL] = (
    "candidate predictor; proxy-risk policy unresolved"
)

column_roles = column_roles.rename_axis("Column").to_frame()

# Country is handled separately because it is always the grouping variable.
candidate_predictor_columns = column_roles.index[
    column_roles["Role"].str.startswith("candidate predictor")
].tolist()

display(column_roles)
display(
    column_roles["Role"]
    .value_counts()
    .rename("Column count")
    .to_frame()
)

# Enforce the assignment-specific role contract.
assert column_roles.index.is_unique
assert set(column_roles.index) == set(train_normalised.columns)
assert not {
    ID_COL,
    GROUP_COL,
    TARGET_COL,
}.intersection(candidate_predictor_columns)
assert CASE_COUNT_COL in candidate_predictor_columns

print(
    f"Role contract: PASS. {len(candidate_predictor_columns)} provisional "
    "non-group candidate predictors were identified; RecordID is excluded "
    "from model inputs and Country is reserved as the grouping variable."
)

,Role
Column,
RecordID,identifier / output mapping only
TARGET_Capacity,regression target
CaseCount,candidate predictor; proxy-risk policy unresolved
Country,grouping variable; feature policy unresolved
Year,candidate predictor
Status,candidate predictor
SystemFailureRate,candidate predictor
ModelFailureRate-T,candidate predictor
ModelFailureRate-NT,candidate predictor


,Column count
Role,
candidate predictor,22
identifier / output mapping only,1
regression target,1
candidate predictor; proxy-risk policy unresolved,1
grouping variable; feature policy unresolved,1


Role contract: PASS. 23 provisional non-group candidate predictors were identified; RecordID is excluded from model inputs and Country is reserved as the grouping variable.


## 2.5 Missingness and supervised-target eligibility

Missingness is audited on the normalised training data without imputing or
altering any values. It is reported by column, by affected row, and by
`Country` and `Year` to distinguish feature-level sparsity from structural
concentration.

Rows without an observed `TARGET_Capacity` are retained in the audit record but
excluded from the supervised modelling pool. Rows with missing predictors
remain eligible; any required imputation is fitted within each training fold.

### 2.5.1 Column-level missingness

In [23]:
missing_by_column = pd.DataFrame({
    "Role": column_roles["Role"],
    "Missing rows": train_normalised.isna().sum(),
    "Missing rate (%)": train_normalised.isna().mean().mul(100).round(3),
})

missing_by_column = (
    missing_by_column.query("`Missing rows` > 0")
    .sort_values("Missing rows", ascending=False)
)

predictor_missing = train_normalised[candidate_predictor_columns].isna()
row_has_missing_predictor = predictor_missing.any(axis=1)

display(missing_by_column)

print(
    f"{int(predictor_missing.sum().sum())} missing predictor cells "
    f"occur across {int(row_has_missing_predictor.sum())} rows."
)

,Role,Missing rows,Missing rate (%)
CaseCount,candidate predictor; proxy-risk policy unresolved,2,0.0970
Complexity,candidate predictor,2,0.0970
Year,candidate predictor,1,0.0480
TARGET_Capacity,regression target,1,0.0480
ModelFailureRate-NT,candidate predictor,1,0.0480
AIDevExpPercent,candidate predictor,1,0.0480
ActiveUserBase,candidate predictor,1,0.0480
DatasetDiversityScore,candidate predictor,1,0.0480


9 missing predictor cells occur across 9 rows.


### 2.5.2 Row-level missingness

In [26]:
all_missing = train_normalised.isna()

missing_by_row = train_normalised[
    [ID_COL, GROUP_COL, "Year"]
].copy()

missing_by_row["Missing count"] = all_missing.sum(axis=1)
missing_by_row["Missing columns"] = all_missing.apply(
    lambda row: ", ".join(row.index[row]),
    axis=1,
)

missing_by_row = (
    missing_by_row.query("`Missing count` > 0")
    .set_index(ID_COL)
)

display(missing_by_row)

print(
    f"{len(missing_by_row)} rows contain missing values; "
    f"{int(row_has_missing_predictor.sum())} contain missing predictors."
)

,Country,Year,Missing count,Missing columns
RecordID,,,,
89,15,NaN,1,Year
170,176,"2,008.0000",1,ModelFailureRate-NT
232,117,"2,010.0000",1,ActiveUserBase
298,170,"2,008.0000",1,DatasetDiversityScore
314,27,"2,008.0000",1,Complexity
315,27,"2,007.0000",1,AIDevExpPercent
319,27,"2,003.0000",1,CaseCount
331,95,"2,007.0000",1,TARGET_Capacity
943,43,"2,004.0000",1,CaseCount


10 rows contain missing values; 9 contain missing predictors.


### 2.5.3 Missingness by Country and Year

In [24]:
country_rows = (
    train_normalised[[GROUP_COL]]
    .rename(columns={GROUP_COL: "Group value"})
    .assign(
        Grouping="Country",
        Missing_row=row_has_missing_predictor.to_numpy(),
    )
)

year_rows = (
    train_normalised[["Year"]]
    .rename(columns={"Year": "Group value"})
    .assign(
        Grouping="Year",
        Missing_row=row_has_missing_predictor.to_numpy(),
    )
)

missing_by_group = (
    pd.concat([country_rows, year_rows], ignore_index=True)
    .groupby(["Grouping", "Group value"], dropna=False)["Missing_row"]
    .agg(Rows="size", Missing_rows="sum")
)

missing_by_group["Missing-row rate (%)"] = (
    missing_by_group["Missing_rows"]
    .div(missing_by_group["Rows"])
    .mul(100)
    .round(2)
)

missing_by_group = (
    missing_by_group.query("Missing_rows > 0")
    .sort_values(
        ["Grouping", "Missing_rows", "Missing-row rate (%)"],
        ascending=[True, False, False],
    )
)

display(missing_by_group)

Rows  Missing_rows  Missing-row rate (%)
Grouping Group value                                          
Country  27.0000        16             3               18.7500
         15.0000        16             1                6.2500
         43.0000        16             1                6.2500
         111.0000       16             1                6.2500
         117.0000       16             1                6.2500
         170.0000       16             1                6.2500
         176.0000       16             1                6.2500
Year     2,008.0000    129             3                2.3300
         NaN             1             1              100.0000
         2,003.0000    129             1                0.7800
         2,004.0000    129             1                0.7800
         2,007.0000    129             1                0.7800
         2,010.0000    129             1                0.7800
         2,012.0000    129             1                0.7800

### 2.5.4 Supervised-target eligibility

In [27]:
missing_target_mask = train_normalised[TARGET_COL].isna()

excluded_unlabelled_train_rows = train_normalised.loc[
    missing_target_mask
].copy()

labelled_data = train_normalised.loc[
    ~missing_target_mask
].copy()

target_eligibility = pd.Series({
    "Input training rows": len(train_normalised),
    "Missing-target rows excluded": len(excluded_unlabelled_train_rows),
    "Supervised-eligible rows": len(labelled_data),
}, name="Rows")

display(target_eligibility.to_frame())

assert labelled_data[TARGET_COL].notna().all()
assert excluded_unlabelled_train_rows[TARGET_COL].isna().all()
assert (
    len(labelled_data) + len(excluded_unlabelled_train_rows)
    == len(train_normalised)
)

print(
    "Target eligibility: PASS. "
    f"{len(excluded_unlabelled_train_rows)} missing-target row was excluded "
    "without target imputation."
)

,Rows
Input training rows,2071
Missing-target rows excluded,1
Supervised-eligible rows,2070


Target eligibility: PASS. 1 missing-target row was excluded without target imputation.


Dataframe usage: Use labelled_data for target-dependent EDA, train/validation splitting, cross-validation, and model training. Keep train_normalised for full-dataset structural audits. excluded_unlabelled_train_rows is retained for audit only, while test_normalised is reserved for final inference.

## 2.6 Duplicate, identifier and key-integrity audit

This section audits duplicate records and key integrity using the full normalised training data, including the row without an observed target. External data are accessed only to verify identifier completeness, uniqueness, and prediction-template order; no external feature values are inspected.

In [28]:
country_year_cols = [GROUP_COL, "Year"]
complete_key_mask = train_normalised[country_year_cols].notna().all(axis=1)

duplicate_rows_without_id = (
    train_normalised.drop(columns=ID_COL).duplicated().sum()
)
duplicate_country_year = (
    train_normalised.loc[complete_key_mask]
    .duplicated(country_year_cols)
    .sum()
)

training_key_audit = pd.Series({
    "Duplicate rows excluding RecordID": duplicate_rows_without_id,
    "Missing RecordID values": train_normalised[ID_COL].isna().sum(),
    "Duplicate RecordID values": train_normalised[ID_COL].duplicated().sum(),
    "Missing Country values": train_normalised[GROUP_COL].isna().sum(),
    "Duplicate complete Country-Year keys": duplicate_country_year,
    "Incomplete Country-Year keys": (~complete_key_mask).sum(),
}, name="Count", dtype="int64")

display(training_key_audit.to_frame())

incomplete_country_year_keys = train_normalised.loc[
    ~complete_key_mask,
    [ID_COL, GROUP_COL, "Year"],
]
display(incomplete_country_year_keys)

assert training_key_audit.drop("Incomplete Country-Year keys").eq(0).all()

print(
    f"Training duplicate/key audit: PASS; "
    f"{len(incomplete_country_year_keys)} incomplete Country-Year key(s) recorded."
)

,Count
Duplicate rows excluding RecordID,0
Missing RecordID values,0
Duplicate RecordID values,0
Missing Country values,0
Duplicate complete Country-Year keys,0
Incomplete Country-Year keys,1


,RecordID,Country,Year
88,89,15,NaN


Training duplicate/key audit: PASS; 1 incomplete Country-Year key(s) recorded.


In [29]:
template_id_order_matches = np.array_equal(
    template_source["ID"].to_numpy(),
    test_normalised[ID_COL].to_numpy(),
)

external_id_audit = pd.Series({
    "Missing test RecordID values": test_normalised[ID_COL].isna().sum(),
    "Duplicate test RecordID values": test_normalised[ID_COL].duplicated().sum(),
    "Missing template ID values": template_source["ID"].isna().sum(),
    "Duplicate template ID values": template_source["ID"].duplicated().sum(),
}, name="Count", dtype="int64")

display(external_id_audit.to_frame())
print(f"Template/test ID order matches: {template_id_order_matches}")

assert external_id_audit.eq(0).all()
assert template_id_order_matches

print("External ID and prediction-template alignment: PASS.")

,Count
Missing test RecordID values,0
Duplicate test RecordID values,0
Missing template ID values,0
Duplicate template ID values,0


Template/test ID order matches: True
External ID and prediction-template alignment: PASS.


In [ ]:
# ============================================================================
# NOTEBOOK STEP 2.7 — Dtypes, cardinality, constants, ranges and structural rules
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/03_data_understanding_and_validation.md
# Workflow heading: Phase 03 > Core tasks checklist > Values, ranges and rules
# Checklist: Check unique counts, constants, unexpected values, ranges and impossible values.
# Data scope: Labelled training data only.
# Expected output / gate: Profiles expose constants/low-cardinality fields and numeric support before modelling.
# ============================================================================

value_profile = pd.DataFrame(
    {
        "dtype": train_raw.dtypes.astype(str),
        "non_missing": train_raw.notna().sum(),
        "unique": train_raw.nunique(dropna=True),
    }
)
value_profile["is_constant"] = value_profile["unique"].le(1)
value_profile["is_low_cardinality"] = value_profile["unique"].le(15)
display(value_profile)

train_structural_numeric = (
    train_raw.drop(columns=[TARGET_COL]).select_dtypes(include=np.number)
)
training_numeric_ranges = train_structural_numeric.agg(
    ["min", "max", "median"]
).T
display(training_numeric_ranges)

low_cardinality_values = {
    col: sorted(train_raw[col].dropna().unique().tolist())
    for col in train_raw.columns
    if train_raw[col].nunique(dropna=True) <= 15
}
print(json.dumps(low_cardinality_values, indent=2, default=str))


In [ ]:
# ============================================================================
# NOTEBOOK STEP 2.8 — Country–Year coverage and group structure
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/02_data_collection_and_governance.md
# - Machine Learning tổng/machine_learning_lifecycle_phases/03_data_understanding_and_validation.md
# Workflow heading: Coverage of groups/time periods and patterns by entity/time
# Checklist: Measure group size, year span, unique years and gaps.
# Data scope: Labelled training data only; target is not aggregated.
# Expected output / gate: Country coverage supports or challenges the grouped-split design.
# ============================================================================

country_year_coverage = (
    train_raw.groupby(GROUP_COL, dropna=False)
    .agg(
        rows=(ID_COL, "size"),
        first_year=("Year", "min"),
        last_year=("Year", "max"),
        unique_years=("Year", "nunique"),
    )
)
country_year_coverage["possible_years_in_span"] = (
    country_year_coverage["last_year"]
    - country_year_coverage["first_year"]
    + 1
)
country_year_coverage["year_gaps"] = (
    country_year_coverage["possible_years_in_span"]
    - country_year_coverage["unique_years"]
)
display(country_year_coverage.describe().T)
display(
    country_year_coverage.sort_values(
        ["year_gaps", "rows"], ascending=[False, True]
    ).head(15)
)
print("Labelled Country count:", train_raw[GROUP_COL].nunique())


In [ ]:
# ============================================================================
# NOTEBOOK STEP 2.9 — Domain exceptions and repeatable validation suite
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/03_data_understanding_and_validation.md
# Workflow heading: Issue log, leakage variables and confirmed-rule tests
# Checklist: Flag anomalous Status/labels and convert confirmed rules into assertions.
# Data scope: Labelled training data only.
# Expected output / gate: Each anomaly has reproducible row evidence; schema validation can be rerun.
# ============================================================================

domain_flags = {
    "status_outside_metadata_0_1_train": train_raw.loc[
        ~train_raw["Status"].isin([0, 1]), [ID_COL, "Status"]
    ],
    "missing_target_train": train_raw.loc[
        train_raw[TARGET_COL].isna(),
        [ID_COL, GROUP_COL, "Year", TARGET_COL],
    ],
}
for flag_name, flagged_rows in domain_flags.items():
    print(f"\n{flag_name}: {len(flagged_rows)} row(s)")
    display(flagged_rows)

def validate_labelled_schema(frame: pd.DataFrame) -> None:
    required = {ID_COL, GROUP_COL, TARGET_COL, CASE_COUNT_COL, "Year", "Status"}
    assert required.issubset(frame.columns)
    assert frame.columns.is_unique
    assert frame[ID_COL].notna().all() and frame[ID_COL].is_unique
    assert frame[GROUP_COL].notna().all()
    assert pd.api.types.is_numeric_dtype(frame[TARGET_COL])

validate_labelled_schema(train_raw)
print(
    "External ranges/categories remain unopened until post-freeze Phase 7. "
    "The target will never be imputed."
)


## 2.10 Data-quality decision log

| Issue | Evidence | Candidate actions | Leakage/bias risk | Final action |
|---|---|---|---|---|
| Missing target | Step 2.5/2.9 | Exclude from supervised fitting | Imputation fabricates ground truth | **Planned:** exclude; TODO confirm |
| Missing predictors | Step 2.5 | Median imputation inside each fold | Global imputation leaks validation information | **Planned:** fold-local |
| `Status=15` despite 0/1 metadata | Step 2.7/2.9 | Retain versus mark missing | Silent correction injects assumptions | TODO ablation |
| `Country` is a numeric repeated identifier | Step 2.8 | group-only versus one-hot | Numeric order is arbitrary | TODO ablation |
| `CaseCount` may be a proxy | Phase 3 | Include/exclude/offset baseline | Shortcut dependence | TODO legitimacy + ablation |
| External support may shift | General risk | Predeclare robustness behaviour | Test-informed clipping contaminates development | TODO freeze before Phase 7 |
| `RecordID` | Step 2.4 | Output mapping only | Spurious identifier pattern | **Fixed:** exclude |

### Phase 2 decision gate

- [ ] Schema and ID assertions pass.
- [ ] Missing target is excluded rather than imputed.
- [ ] Every anomaly has a written, testable policy.
- [ ] External value distributions have not influenced development.
- [ ] `RecordID` is excluded from every model matrix.

**Decision:** TODO — state which rows/values are retained, transformed or
excluded and the evidence for each choice.


# PHASE 3 — Grouped Split & Development-Only EDA

**Lifecycle coverage:** `04_data_splitting.md` + target-informed EDA from `03_data_understanding_and_validation.md`

> **Workflow adaptation:** The holdout is created before any target-informed EDA. Every plot/table from Step 3.3 onward uses development_df only.

| Notebook step | Concrete purpose | Exact workflow file |
|---|---|---|
| 3.1 | Create Country-grouped internal holdout | `04_data_splitting.md` |
| 3.2 | Lock/validate split and save an in-memory manifest | `04_data_splitting.md` |
| 3.3 | Inspect development target distribution | `03_data_understanding_and_validation.md` |
| 3.4 | Profile univariate predictors | `03_data_understanding_and_validation.md` |
| 3.5 | Inspect development Country/Year patterns | `03_data_understanding_and_validation.md` |
| 3.6 | Examine feature–target relationships | `03_data_understanding_and_validation.md` |
| 3.7 | Investigate CaseCount legitimacy/offset | `03_data_understanding_and_validation.md` |
| 3.8 | Screen predictor–predictor collinearity | `03_data_understanding_and_validation.md` |
| 3.9 | Define training support and stress-test plan | `10_error_analysis_and_robustness.md` |
| 3.10 | Translate EDA evidence into experiments | `03_data_understanding_and_validation.md` |

Every code cell below repeats its workflow source, data scope, and expected
output/gate in comments. Heavy training, holdout access, and file export remain
behind explicit flags.


In [ ]:
holdout_group_fraction = (
    holdout_df[GROUP_COL].nunique()
    / labelled_data[GROUP_COL].nunique()
)

holdout_row_fraction = len(holdout_df) / len(labelled_data)

print(f"Holdout group fraction: {holdout_group_fraction:.2%}")
print(f"Holdout row fraction:   {holdout_row_fraction:.2%}") 
evidence for 0.20 in phase 0.6
The grouped split reserved 28 of 136 countries (20.59%) and 417 of 2,070
labelled rows (20.14%). The row proportion is close to the intended 20%
because most countries contain a similar number of observations.

In [ ]:
# ============================================================================
# NOTEBOOK STEP 3.1 — Create the Country-grouped internal holdout
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/04_data_splitting.md
# Workflow heading: Phase 04 > Core tasks checklist
# Checklist: Define independent group; reserve a test/holdout; keep each group in one partition.
# Data scope: Labelled rows only; missing-target rows are excluded and logged.
# Expected output / gate: development_df and holdout_df are created using the frozen seed.
# ============================================================================

labelled_data = train_raw.loc[train_raw[TARGET_COL].notna()].copy()
excluded_unlabelled_train_rows = train_raw.loc[
    train_raw[TARGET_COL].isna()
].copy()

group_holdout_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=INTERNAL_HOLDOUT_FRACTION,
    random_state=RANDOM_STATE,
)
development_idx, holdout_idx = next(
    group_holdout_splitter.split(
        labelled_data,
        groups=labelled_data[GROUP_COL],
    )
)
development_df = labelled_data.iloc[development_idx].copy()
holdout_df = labelled_data.iloc[holdout_idx].copy()


In [ ]:
# ============================================================================
# NOTEBOOK STEP 3.2 — Lock, validate and record the split
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/04_data_splitting.md
# Workflow heading: Phase 04 > Core tasks checklist and exit criteria
# Checklist: Verify zero group overlap; save IDs; record splitter, seed, version and sizes.
# Data scope: Only holdout row/group/schema counts are inspected; no holdout target summary.
# Expected output / gate: Zero Country overlap, enough development groups, and an immutable split manifest.
# ============================================================================

development_countries = set(development_df[GROUP_COL])
holdout_countries = set(holdout_df[GROUP_COL])
assert development_countries.isdisjoint(holdout_countries)
assert development_df[GROUP_COL].nunique() >= N_GROUP_FOLDS

split_summary = pd.DataFrame(
    {
        "rows": [len(development_df), len(holdout_df)],
        "countries": [
            development_df[GROUP_COL].nunique(),
            holdout_df[GROUP_COL].nunique(),
        ],
        "target_missing": [
            development_df[TARGET_COL].isna().sum(),
            holdout_df[TARGET_COL].isna().sum(),
        ],
    },
    index=["development", "locked_holdout"],
)
split_manifest = {
    "splitter": "GroupShuffleSplit",
    "group_column": GROUP_COL,
    "test_size": INTERNAL_HOLDOUT_FRACTION,
    "random_state": RANDOM_STATE,
    "train_sha256": source_inventory.loc[
        source_inventory["relative_path"].str.endswith("train.csv"), "sha256"
    ].iloc[0],
    "development_record_ids": development_df[ID_COL].tolist(),
    "holdout_record_ids": holdout_df[ID_COL].tolist(),
}
display(split_summary)
print("Country overlap:", len(development_countries & holdout_countries))
print("Excluded missing-target rows:", len(excluded_unlabelled_train_rows))
print(
    "Split-manifest SHA256:",
    hashlib.sha256(
        json.dumps(split_manifest, sort_keys=True).encode("utf-8")
    ).hexdigest(),
)


> **Locked-holdout rule:** until Phase 7, do not calculate holdout target
> summaries, correlations, plots, metrics, residuals, slice results or tuning
> decisions. Only its row/group counts, IDs and schema may be checked.


In [ ]:
# ============================================================================
# NOTEBOOK STEP 3.3 — Development target distribution
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/03_data_understanding_and_validation.md
# Workflow heading: Phase 03 > Univariate EDA
# Checklist: Inspect target scale, skew, spread and possible outliers.
# Data scope: development_df only; locked holdout and external values are excluded.
# Expected output / gate: Summary and plots support metric/transform decisions.
# ============================================================================

target_summary = development_df[TARGET_COL].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)
display(target_summary.to_frame("development_target"))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(development_df[TARGET_COL], bins=30, edgecolor="white")
axes[0].set(
    title="Development target distribution",
    xlabel=TARGET_COL,
    ylabel="Count",
)
axes[1].boxplot(development_df[TARGET_COL], orientation="horizontal")
axes[1].set(title="Development target boxplot", xlabel=TARGET_COL)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================================
# NOTEBOOK STEP 3.4 — Univariate predictor distributions
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/03_data_understanding_and_validation.md
# Workflow heading: Phase 03 > Univariate EDA
# Checklist: Inspect numeric distributions, skew, missingness and unusual support.
# Data scope: development_df predictors only.
# Expected output / gate: A profile and limited diagnostic histograms identify transformations worth testing.
# ============================================================================

eda_numeric_features = [
    col
    for col in development_df.select_dtypes(include=np.number).columns
    if col not in {ID_COL, TARGET_COL, GROUP_COL}
]
univariate_profile = development_df[eda_numeric_features].describe(
    percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
).T
univariate_profile["missing"] = development_df[eda_numeric_features].isna().sum()
univariate_profile["skew"] = development_df[eda_numeric_features].skew(numeric_only=True)
display(univariate_profile)

plot_features = eda_numeric_features[: min(6, len(eda_numeric_features))]
if plot_features:
    fig, axes = plt.subplots(
        len(plot_features), 1, figsize=(9, 2.6 * len(plot_features))
    )
    for axis, feature in zip(np.atleast_1d(axes), plot_features):
        axis.hist(
            development_df[feature].dropna(),
            bins=30,
            edgecolor="white",
        )
        axis.set(title=f"Development distribution: {feature}", xlabel=feature)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric predictor is available for univariate plotting.")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 3.5 — Development Country and Year patterns
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/03_data_understanding_and_validation.md
# Workflow heading: Phase 03 > Patterns by time, entity and group
# Checklist: Check temporal/group coverage and target behaviour within development only.
# Data scope: development_df only.
# Expected output / gate: Year and Country tables expose imbalance/trends without touching the holdout.
# ============================================================================

development_year_profile = (
    development_df.groupby("Year", dropna=False)
    .agg(
        rows=(ID_COL, "size"),
        countries=(GROUP_COL, "nunique"),
        target_mean=(TARGET_COL, "mean"),
        target_median=(TARGET_COL, "median"),
    )
    .sort_index()
)
development_country_profile = (
    development_df.groupby(GROUP_COL, dropna=False)
    .agg(
        rows=(ID_COL, "size"),
        first_year=("Year", "min"),
        last_year=("Year", "max"),
        target_mean=(TARGET_COL, "mean"),
        target_std=(TARGET_COL, "std"),
    )
)
display(development_year_profile)
display(development_country_profile.describe().T)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    development_year_profile.index,
    development_year_profile["target_median"],
    marker="o",
)
ax.set(
    title="Development median target by Year",
    xlabel="Year",
    ylabel=f"Median {TARGET_COL}",
)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================================
# NOTEBOOK STEP 3.6 — Feature–target relationship screen
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/03_data_understanding_and_validation.md
# Workflow heading: Phase 03 > Bivariate EDA
# Checklist: Examine plausible predictors with Pearson and Spearman evidence.
# Data scope: development_df only.
# Expected output / gate: Association table suggests—not automatically selects—linear/nonlinear hypotheses.
# ============================================================================

development_numeric = (
    development_df.drop(columns=[ID_COL, GROUP_COL])
    .select_dtypes(include=np.number)
)
pearson_with_target = (
    development_numeric.corr(method="pearson", numeric_only=True)[TARGET_COL]
    .drop(labels=[TARGET_COL])
)
spearman_with_target = (
    development_numeric.corr(method="spearman", numeric_only=True)[TARGET_COL]
    .drop(labels=[TARGET_COL])
)
target_association = (
    pd.DataFrame(
        {
            "pearson": pearson_with_target,
            "spearman": spearman_with_target,
        }
    )
    .assign(max_abs=lambda frame: frame.abs().max(axis=1))
    .sort_values("max_abs", ascending=False)
)
display(target_association)
print(
    "Correlation is exploratory evidence only; no feature is deleted "
    "automatically from this table."
)


In [ ]:
# ============================================================================
# NOTEBOOK STEP 3.7 — CaseCount legitimacy and offset-baseline premise
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/03_data_understanding_and_validation.md
# - Machine Learning tổng/machine_learning_lifecycle_phases/06_baseline_development.md
# Workflow heading: Leakage-variable audit / problem-specific baseline
# Checklist: Investigate possible proxy dependence and define a transparent comparator.
# Data scope: development_df only.
# Expected output / gate: Correlation, offset spread and plots justify a later include/exclude ablation.
# ============================================================================

case_target_corr = development_df[[CASE_COUNT_COL, TARGET_COL]].corr().iloc[0, 1]
case_offset = development_df[TARGET_COL] - development_df[CASE_COUNT_COL]
print(
    f"Development Pearson correlation "
    f"({CASE_COUNT_COL}, {TARGET_COL}): {case_target_corr:.6f}"
)
display(case_offset.describe().to_frame("TARGET_Capacity_minus_CaseCount"))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(
    development_df[CASE_COUNT_COL],
    development_df[TARGET_COL],
    alpha=0.45,
    s=18,
)
axes[0].set(
    title="Development: target vs CaseCount",
    xlabel=CASE_COUNT_COL,
    ylabel=TARGET_COL,
)
axes[1].scatter(
    development_df[CASE_COUNT_COL], case_offset, alpha=0.45, s=18
)
axes[1].axhline(
    case_offset.mean(), color="tab:red", linestyle="--", label="Mean offset"
)
axes[1].set(
    title="Residual from CaseCount",
    xlabel=CASE_COUNT_COL,
    ylabel="Target − CaseCount",
)
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================================
# NOTEBOOK STEP 3.8 — Predictor–predictor multicollinearity screen
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/03_data_understanding_and_validation.md
# Workflow heading: Phase 03 > Multivariate EDA
# Checklist: Identify redundant signals/interactions without automatic feature deletion.
# Data scope: development_df numeric predictors only; target excluded.
# Expected output / gate: Top absolute predictor-pair correlations motivate Ridge/Lasso and stability checks.
# ============================================================================

collinearity_features = [
    col
    for col in development_df.select_dtypes(include=np.number).columns
    if col not in {ID_COL, TARGET_COL, GROUP_COL}
]
predictor_correlation = development_df[collinearity_features].corr(
    numeric_only=True
)
upper_triangle = predictor_correlation.where(
    np.triu(np.ones(predictor_correlation.shape), k=1).astype(bool)
)
strongest_pairs = (
    upper_triangle.stack()
    .rename("correlation")
    .to_frame()
    .assign(abs_correlation=lambda frame: frame["correlation"].abs())
    .sort_values("abs_correlation", ascending=False)
    .head(20)
)
display(strongest_pairs)


In [ ]:
# ============================================================================
# NOTEBOOK STEP 3.9 — Development support and extrapolation-risk register
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/03_data_understanding_and_validation.md
# - Machine Learning tổng/machine_learning_lifecycle_phases/10_error_analysis_and_robustness.md
# Workflow heading: Production-input compatibility / robustness stress tests
# Checklist: Freeze training-derived support and plausible perturbations before external values open.
# Data scope: development_df only.
# Expected output / gate: Quantile/support table and stress scenarios become Phase 6/7 reference evidence.
# ============================================================================

support_features = [
    col
    for col in development_df.select_dtypes(include=np.number).columns
    if col not in {ID_COL, TARGET_COL, GROUP_COL}
]
development_support = (
    development_df[support_features]
    .quantile([0.00, 0.01, 0.05, 0.50, 0.95, 0.99, 1.00])
    .T
)
development_support.columns = [
    "min",
    "p01",
    "p05",
    "median",
    "p95",
    "p99",
    "max",
]
display(development_support)

predeclared_stress_plan = pd.DataFrame(
    [
        ["missing_numeric", "set one selected numeric feature to NaN", "finite predictions; quantify change"],
        ["high_numeric", "set one selected numeric feature above development max", "finite predictions; flag extrapolation"],
        ["unseen_country", "use a Country category absent from fitting", "OneHot encoder ignores safely"],
        ["CaseCount_shift", "perturb CaseCount if selected", "quantify shortcut sensitivity"],
    ],
    columns=["scenario", "construction", "expected_check"],
)
display(predeclared_stress_plan)
print("External values remain unopened; this register is training-derived.")


## 3.10 EDA-to-model decision table

| Evidence | Interpretation | Modelling consequence | Test |
|---|---|---|---|
| Countries repeat across years | Row-wise folds share entities | GroupShuffleSplit + GroupKFold | Fold/group manifest |
| `CaseCount` is close to target | Legitimate driver or proxy risk | Offset baseline + with/without ablation | Steps 4.7 and 6.7 |
| TODO target scale/skew | TODO | Metric/transform implication | Fixed grouped CV |
| TODO collinearity | TODO | Ridge/Lasso may stabilise | Coefficients + fold stability |
| TODO nonlinear evidence | TODO | Polynomial/tree candidate | Shared grouped CV |

### Phase 3 decision gate

- [ ] No target-informed inspection used the locked holdout or external test.
- [ ] Every selected transform/model family is linked to development evidence.
- [ ] CaseCount legitimacy and ablation plan are recorded.
- [ ] Split seed and IDs are frozen.

**Decision:** TODO — record the 3–5 EDA findings that materially change the
modelling plan.


# PHASE 4 — Preprocessing, Policy Ablations & Baselines

**Lifecycle coverage:** `05_preprocessing_and_feature_engineering.md` + `06_baseline_development.md`

| Notebook step | Concrete purpose | Exact workflow file |
|---|---|---|
| 4.1 | Freeze feature-inclusion policy | `05_preprocessing_and_feature_engineering.md` |
| 4.2 | Apply deterministic domain rules | `05_preprocessing_and_feature_engineering.md` |
| 4.3 | Define numeric preprocessing | `05_preprocessing_and_feature_engineering.md` |
| 4.4 | Define categorical handling | `05_preprocessing_and_feature_engineering.md` |
| 4.5 | Build/smoke-test fold-local preprocessors | `05_preprocessing_and_feature_engineering.md` |
| 4.6 | Run fixed-protocol policy ablations | `10_error_analysis_and_robustness.md` |
| 4.7 | Evaluate dummy/problem-specific/simple baselines | `06_baseline_development.md` |
| 4.8 | Conclude baseline stage | `06_baseline_development.md` |

Every code cell below repeats its workflow source, data scope, and expected
output/gate in comments. Heavy training, holdout access, and file export remain
behind explicit flags.


In [ ]:
# ============================================================================
# NOTEBOOK STEP 4.1 — Feature-inclusion policy and provisional feature list
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/05_preprocessing_and_feature_engineering.md
# Workflow heading: Phase 05 > Core tasks checklist
# Checklist: Separate roles and verify prediction-time availability.
# Data scope: Development schema only.
# Expected output / gate: ID/target are excluded; Country is group-only unless a one-hot policy is confirmed.
# ============================================================================

MANDATORY_NON_PREDICTORS = {ID_COL, TARGET_COL}
ALL_CANDIDATE_FEATURES = [
    col for col in development_df.columns if col not in MANDATORY_NON_PREDICTORS
]
ACTIVE_COUNTRY_POLICY = COUNTRY_FEATURE_POLICY or "group_only"
FEATURE_COLUMNS = [
    col
    for col in ALL_CANDIDATE_FEATURES
    if col not in set(ADDITIONAL_FEATURE_EXCLUSIONS)
    and (col != GROUP_COL or ACTIVE_COUNTRY_POLICY == "one_hot")
]
assert ID_COL not in FEATURE_COLUMNS and TARGET_COL not in FEATURE_COLUMNS
print("Provisional features:", FEATURE_COLUMNS)
print(
    "Country policy:",
    ACTIVE_COUNTRY_POLICY,
    "(confirmed:",
    DEVELOPMENT_POLICY_CONFIRMED,
    ")",
)


In [ ]:
# ============================================================================
# NOTEBOOK STEP 4.2 — Deterministic Status rule and model-frame construction
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/05_preprocessing_and_feature_engineering.md
# Workflow heading: Phase 05 > Core tasks checklist
# Checklist: Apply deterministic rules; keep learned imputation inside folds.
# Data scope: Development features only.
# Expected output / gate: X/y/groups are aligned and policy validation blocks unresolved training.
# ============================================================================

def apply_status_rule(frame: pd.DataFrame, policy: str | None) -> pd.DataFrame:
    result = frame.copy()
    if policy is None or policy == "retain_numeric":
        return result
    if policy == "invalid_to_nan":
        result.loc[~result["Status"].isin([0, 1]), "Status"] = np.nan
        return result
    raise ValueError(f"Unsupported Status policy: {policy!r}")

def prepare_feature_frame(
    frame: pd.DataFrame,
    feature_columns: list[str],
    status_policy: str | None,
) -> pd.DataFrame:
    cleaned = apply_status_rule(frame, status_policy)
    missing_columns = sorted(set(feature_columns) - set(cleaned.columns))
    assert not missing_columns, f"Missing required feature(s): {missing_columns}"
    return cleaned[feature_columns].copy()

def validate_development_policy() -> None:
    assert DEVELOPMENT_POLICY_CONFIRMED, "Development policy is not signed off."
    assert STATUS_INVALID_POLICY in {"retain_numeric", "invalid_to_nan"}
    assert COUNTRY_FEATURE_POLICY in {"group_only", "one_hot"}
    invalid_exclusions = (
        set(ADDITIONAL_FEATURE_EXCLUSIONS) - set(ALL_CANDIDATE_FEATURES)
    )
    assert not invalid_exclusions, (
        f"Unknown feature exclusion(s): {sorted(invalid_exclusions)}"
    )
    assert GROUP_COL not in ADDITIONAL_FEATURE_EXCLUSIONS

X_development = prepare_feature_frame(
    development_df, FEATURE_COLUMNS, STATUS_INVALID_POLICY
)
y_development = development_df[TARGET_COL].copy()
groups_development = development_df[GROUP_COL].copy()
assert X_development.index.equals(y_development.index)


In [ ]:
# ============================================================================
# NOTEBOOK STEP 4.3 — Numeric preprocessing factory
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/05_preprocessing_and_feature_engineering.md
# Workflow heading: Phase 05 > Core tasks checklist
# Checklist: Handle missing values and scale only where useful; fit learned steps within folds.
# Data scope: Definition only; no fitting.
# Expected output / gate: Reusable numeric Pipeline contains median imputation and optional scaling.
# ============================================================================

def make_numeric_pipeline(scale_numeric: bool) -> Pipeline:
    steps = [
        ("imputer", SimpleImputer(strategy="median", add_indicator=False))
    ]
    if scale_numeric:
        steps.append(("scaler", StandardScaler()))
    return Pipeline(steps)


In [ ]:
# ============================================================================
# NOTEBOOK STEP 4.4 — Categorical handling and ColumnTransformer
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/05_preprocessing_and_feature_engineering.md
# Workflow heading: Phase 05 > Core tasks checklist
# Checklist: Encode categories safely and define unseen-category behaviour.
# Data scope: Definition only; no fitting.
# Expected output / gate: Country is either excluded or one-hot encoded with handle_unknown='ignore'.
# ============================================================================

def make_preprocessor(
    feature_columns: list[str],
    country_policy: str,
    scale_numeric: bool,
) -> ColumnTransformer:
    numeric_features = [
        col for col in feature_columns if col != GROUP_COL
    ]
    transformers = [
        (
            "numeric",
            make_numeric_pipeline(scale_numeric),
            numeric_features,
        )
    ]
    if GROUP_COL in feature_columns:
        assert country_policy == "one_hot"
        transformers.append(
            (
                "country",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                [GROUP_COL],
            )
        )
    else:
        assert country_policy == "group_only"
    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=False,
    )


In [ ]:
# ============================================================================
# NOTEBOOK STEP 4.5A — Polynomial numeric preprocessor
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/05_preprocessing_and_feature_engineering.md
# Workflow heading: Phase 05 > Pipeline and feature-engineering checklist
# Checklist: Keep learned transforms inside the Pipeline and avoid dense Country interactions.
# Data scope: Definition only; no fitting.
# Expected output / gate: Numeric polynomial expansion is isolated from optional Country one-hot encoding.
# ============================================================================

def make_polynomial_preprocessor(
    feature_columns: list[str],
    country_policy: str,
) -> ColumnTransformer:
    numeric_features = [
        col for col in feature_columns if col != GROUP_COL
    ]
    numeric_polynomial = Pipeline(
        [
            (
                "imputer",
                SimpleImputer(strategy="median", add_indicator=False),
            ),
            (
                "poly",
                PolynomialFeatures(degree=2, include_bias=False),
            ),
            ("scaler", StandardScaler()),
        ]
    )
    transformers = [
        ("numeric_poly", numeric_polynomial, numeric_features)
    ]
    if GROUP_COL in feature_columns:
        assert country_policy == "one_hot"
        transformers.append(
            (
                "country",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                [GROUP_COL],
            )
        )
    else:
        assert country_policy == "group_only"
    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=False,
    )


In [ ]:
# ============================================================================
# NOTEBOOK STEP 4.5B — Preprocessing shape/name/memory smoke test
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/05_preprocessing_and_feature_engineering.md
# Workflow heading: Phase 05 > Test schema, feature names/order, shape and sparsity
# Checklist: Verify transformed output before launching expensive training.
# Data scope: Development only; gated because it fits preprocessing to the full development set for diagnostics.
# Expected output / gate: Feature count/names/memory are recorded; this output is not a performance estimate.
# ============================================================================

preprocessing_smoke_test = None
if RUN_PREPROCESSING_SMOKE_TEST:
    validate_development_policy()
    smoke_preprocessor = make_preprocessor(
        FEATURE_COLUMNS,
        COUNTRY_FEATURE_POLICY,
        scale_numeric=True,
    )
    smoke_matrix = smoke_preprocessor.fit_transform(X_development)
    smoke_feature_names = smoke_preprocessor.get_feature_names_out()
    preprocessing_smoke_test = pd.Series(
        {
            "input_rows": X_development.shape[0],
            "input_features": X_development.shape[1],
            "output_features": smoke_matrix.shape[1],
            "output_megabytes": smoke_matrix.nbytes / (1024 ** 2),
            "finite_output": bool(np.isfinite(smoke_matrix).all()),
        },
        name="Preprocessing smoke test",
    )
    display(preprocessing_smoke_test)
    display(pd.Series(smoke_feature_names, name="transformed_feature"))
else:
    print("Preprocessing smoke test skipped (safe default).")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 4.6A — Fixed grouped-CV helper for ablations and baselines
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/06_baseline_development.md
# - Machine Learning tổng/machine_learning_lifecycle_phases/08_validation_and_hyperparameter_tuning.md
# Workflow heading: Same evaluation protocol / fair model comparison
# Checklist: Use identical group folds, metrics, timing and train–validation evidence.
# Data scope: Definition only; no fitting.
# Expected output / gate: One helper prevents protocol drift across baseline, candidate and ablation stages.
# ============================================================================

GROUP_CV = GroupKFold(
    n_splits=N_GROUP_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

def compare_models_grouped_cv(
    models,
    X,
    y,
    groups,
    cv,
) -> pd.DataFrame:
    rows = []
    for model_name, estimator in models.items():
        started = time.perf_counter()
        scores = cross_validate(
            estimator,
            X,
            y,
            groups=groups,
            cv=cv,
            scoring=SCORING,
            return_train_score=True,
            n_jobs=N_JOBS,
            error_score="raise",
        )
        rows.append(
            {
                "model": model_name,
                "cv_rmse_mean": -scores["test_rmse"].mean(),
                "cv_rmse_std": scores["test_rmse"].std(ddof=1),
                "train_rmse_mean": -scores["train_rmse"].mean(),
                "train_valid_rmse_gap": (
                    -scores["test_rmse"].mean()
                    + scores["train_rmse"].mean()
                ),
                "cv_mae_mean": -scores["test_mae"].mean(),
                "cv_mae_std": scores["test_mae"].std(ddof=1),
                "train_mae_mean": -scores["train_mae"].mean(),
                "train_valid_mae_gap": (
                    -scores["test_mae"].mean()
                    + scores["train_mae"].mean()
                ),
                "cv_r2_mean": scores["test_r2"].mean(),
                "cv_r2_std": scores["test_r2"].std(ddof=1),
                "train_r2_mean": scores["train_r2"].mean(),
                "train_valid_r2_gap": (
                    scores["train_r2"].mean()
                    - scores["test_r2"].mean()
                ),
                "fit_time_mean": scores["fit_time"].mean(),
                "score_time_mean": scores["score_time"].mean(),
                "runtime_seconds": time.perf_counter() - started,
            }
        )
    table = pd.DataFrame(rows)
    primary_key = validate_primary_metric()
    primary_columns = {
        "rmse": (
            "cv_rmse_mean",
            "cv_rmse_std",
            "train_rmse_mean",
            "train_valid_rmse_gap",
        ),
        "mae": (
            "cv_mae_mean",
            "cv_mae_std",
            "train_mae_mean",
            "train_valid_mae_gap",
        ),
        "r2": (
            "cv_r2_mean",
            "cv_r2_std",
            "train_r2_mean",
            "train_valid_r2_gap",
        ),
    }[primary_key]
    table["primary_cv_mean"] = table[primary_columns[0]]
    table["primary_cv_std"] = table[primary_columns[1]]
    table["primary_train_mean"] = table[primary_columns[2]]
    table["primary_gap"] = table[primary_columns[3]]
    return table.sort_values(
        "primary_cv_mean",
        ascending=primary_key in LOSS_METRICS,
    ).reset_index(drop=True)

print("Shared grouped-CV helper defined; no estimator fitted.")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 4.6B — Country, Status and CaseCount policy ablations
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/03_data_understanding_and_validation.md
# - Machine Learning tổng/machine_learning_lifecycle_phases/10_error_analysis_and_robustness.md
# Workflow heading: Evidence-driven follow-up experiments / feature-data-rule ablations
# Checklist: Compare predeclared policies using one transparent model and fixed group folds.
# Data scope: development_df only; gated.
# Expected output / gate: Policy summary supports—not replaces—the written feature-policy decision.
# ============================================================================

policy_ablation_summary = None
if RUN_POLICY_ABLATIONS:
    validate_primary_metric()
    base_no_country = [
        col
        for col in ALL_CANDIDATE_FEATURES
        if col != GROUP_COL
        and col not in set(ADDITIONAL_FEATURE_EXCLUSIONS)
    ]
    policy_ablation_specs = {
        "group_only_status_retained": {
            "features": base_no_country,
            "country_policy": "group_only",
            "status_policy": "retain_numeric",
        },
        "country_one_hot_status_retained": {
            "features": [
                col
                for col in ALL_CANDIDATE_FEATURES
                if col not in set(ADDITIONAL_FEATURE_EXCLUSIONS)
            ],
            "country_policy": "one_hot",
            "status_policy": "retain_numeric",
        },
        "without_CaseCount": {
            "features": [
                col for col in base_no_country if col != CASE_COUNT_COL
            ],
            "country_policy": "group_only",
            "status_policy": "retain_numeric",
        },
        "invalid_Status_to_nan": {
            "features": base_no_country,
            "country_policy": "group_only",
            "status_policy": "invalid_to_nan",
        },
    }
    policy_rows = []
    for label, policy_spec in policy_ablation_specs.items():
        ablation_X = prepare_feature_frame(
            development_df,
            policy_spec["features"],
            policy_spec["status_policy"],
        )
        ablation_model = Pipeline(
            [
                (
                    "preprocess",
                    make_preprocessor(
                        policy_spec["features"],
                        policy_spec["country_policy"],
                        scale_numeric=True,
                    ),
                ),
                ("model", LinearRegression()),
            ]
        )
        row = compare_models_grouped_cv(
            {label: ablation_model},
            ablation_X,
            y_development,
            groups_development,
            GROUP_CV,
        ).iloc[0].to_dict()
        row.update(policy_spec)
        policy_rows.append(row)
    policy_ablation_summary = pd.DataFrame(policy_rows)
    display(policy_ablation_summary)
else:
    print("Policy ablations skipped; enable before freezing feature policies.")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 4.7A — Problem-specific CaseCount offset baseline
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/06_baseline_development.md
# Workflow heading: Phase 06 > Core tasks checklist
# Checklist: Implement a transparent task-aligned baseline; learn its offset inside each fold.
# Data scope: Estimator definition only.
# Expected output / gate: Baseline never estimates its offset globally before CV.
# ============================================================================

class CaseCountOffsetRegressor(RegressorMixin, BaseEstimator):
    def __init__(self, case_count_col: str = CASE_COUNT_COL):
        self.case_count_col = case_count_col

    def fit(self, X: pd.DataFrame, y):
        case_count = pd.to_numeric(
            X[self.case_count_col], errors="coerce"
        )
        self.case_count_median_ = float(case_count.median())
        filled = case_count.fillna(self.case_count_median_).to_numpy()
        self.offset_ = float(np.mean(np.asarray(y) - filled))
        self.n_features_in_ = X.shape[1]
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        if not hasattr(self, "offset_"):
            raise RuntimeError("Fit CaseCountOffsetRegressor before predict().")
        case_count = pd.to_numeric(
            X[self.case_count_col], errors="coerce"
        )
        return (
            case_count.fillna(self.case_count_median_).to_numpy()
            + self.offset_
        )

def grouped_case_count_offset_cv(
    frame: pd.DataFrame,
    cv: GroupKFold,
) -> pd.DataFrame:
    records = []
    X = frame[[CASE_COUNT_COL]]
    y = frame[TARGET_COL]
    groups = frame[GROUP_COL]
    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X, y, groups), start=1
    ):
        fold_train = frame.iloc[train_idx]
        fold_valid = frame.iloc[valid_idx]
        median = fold_train[CASE_COUNT_COL].median()
        train_case = fold_train[CASE_COUNT_COL].fillna(median)
        valid_case = fold_valid[CASE_COUNT_COL].fillna(median)
        offset = (fold_train[TARGET_COL] - train_case).mean()
        predictions = valid_case + offset
        records.append(
            {
                "fold": fold,
                "learned_offset": offset,
                "fold_case_median": median,
                "validation_values_imputed": int(
                    fold_valid[CASE_COUNT_COL].isna().sum()
                ),
                "rmse": root_mean_squared_error(
                    fold_valid[TARGET_COL], predictions
                ),
                "mae": mean_absolute_error(
                    fold_valid[TARGET_COL], predictions
                ),
                "r2": r2_score(fold_valid[TARGET_COL], predictions),
            }
        )
    return pd.DataFrame(records)


In [ ]:
# ============================================================================
# NOTEBOOK STEP 4.7B — Dummy, problem-specific and simple linear baseline evaluation
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/06_baseline_development.md
# - Machine Learning tổng/machine_learning_lifecycle_phases/09_evaluation_metrics.md
# Workflow heading: Baseline core tasks / fair comparison
# Checklist: Evaluate baselines under the same grouped folds, metrics and timing protocol.
# Data scope: development_df only; gated.
# Expected output / gate: Fold results, means, spread, gaps and runtime establish minimum performance.
# ============================================================================

baseline_results = None
case_offset_fold_results = None
if RUN_BASELINE_EVALUATION:
    validate_primary_metric()
    validate_development_policy()
    baseline_models = {
        "DummyMean": Pipeline(
            [
                (
                    "preprocess",
                    make_preprocessor(
                        FEATURE_COLUMNS,
                        COUNTRY_FEATURE_POLICY,
                        scale_numeric=False,
                    ),
                ),
                ("model", DummyRegressor(strategy="mean")),
            ]
        ),
        "LinearRegression_W2": Pipeline(
            [
                (
                    "preprocess",
                    make_preprocessor(
                        FEATURE_COLUMNS,
                        COUNTRY_FEATURE_POLICY,
                        scale_numeric=True,
                    ),
                ),
                ("model", LinearRegression()),
            ]
        ),
    }
    if CASE_COUNT_COL in FEATURE_COLUMNS:
        baseline_models["CaseCountOffset"] = CaseCountOffsetRegressor()
        case_offset_fold_results = grouped_case_count_offset_cv(
            development_df, GROUP_CV
        )
        display(case_offset_fold_results)
    baseline_results = compare_models_grouped_cv(
        baseline_models,
        X_development,
        y_development,
        groups_development,
        GROUP_CV,
    )
    display(baseline_results)
else:
    print("Baseline evaluation skipped (safe default).")


## 4.8 Baseline conclusion

**TODO:** cite grouped-fold primary mean ± SD, secondary metrics,
train–validation gap and runtime. State which baselines are beaten and what
minimum improvement justifies more complex candidates.

### Phase 4 decision gate

- [ ] Learned preprocessing is inside pipelines/folds.
- [ ] Feature names/shape/memory smoke test is recorded.
- [ ] Country, Status and CaseCount policies are evidence-based and frozen.
- [ ] Dummy, CaseCount-offset and simple linear baselines use identical folds.
- [ ] `DEVELOPMENT_POLICY_CONFIRMED=True` only after the written decision.


# PHASE 5 — Candidate Comparison, Tuning & Finalist Freeze

**Lifecycle coverage:** `07_model_training.md` + `08_validation_and_hyperparameter_tuning.md` + `09_evaluation_metrics.md`

| Notebook step | Concrete purpose | Exact workflow file |
|---|---|---|
| 5.1 | Define a small justified candidate family set | `07_model_training.md` |
| 5.2 | Record the shared grouped-CV fold protocol | `08_validation_and_hyperparameter_tuning.md` |
| 5.3 | Compare initial configurations | `07_model_training.md` |
| 5.4 | Select only justified families for tuning | `08_validation_and_hyperparameter_tuning.md` |
| 5.5 | Tune Week 3 regularised models | `08_validation_and_hyperparameter_tuning.md` |
| 5.6 | Tune Week 5 regression extension | `08_validation_and_hyperparameter_tuning.md` |
| 5.7 | Tune beyond-class model | `08_validation_and_hyperparameter_tuning.md` |
| 5.8 | Consolidate fair comparison evidence | `09_evaluation_metrics.md` |
| 5.9 | Inspect folds and generalisation gaps | `08_validation_and_hyperparameter_tuning.md` |
| 5.10 | Freeze multiple finalists | `11_final_evaluation.md` |

Every code cell below repeats its workflow source, data scope, and expected
output/gate in comments. Heavy training, holdout access, and file export remain
behind explicit flags.


In [ ]:
# ============================================================================
# NOTEBOOK STEP 5.1 — Candidate pipelines and rationale
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/07_model_training.md
# Workflow heading: Phase 07 > Core tasks checklist
# Checklist: Use a small justified family set; record objectives, capacity controls and risks.
# Data scope: Definitions only; no fitting.
# Expected output / gate: Every candidate is a complete reproducible estimator/pipeline with a course link.
# ============================================================================

def make_candidate_estimators(
    feature_columns: list[str],
    country_policy: str,
):
    scaled = make_preprocessor(
        feature_columns, country_policy, scale_numeric=True
    )
    unscaled = make_preprocessor(
        feature_columns, country_policy, scale_numeric=False
    )
    estimators = {
        "DummyMean": Pipeline(
            [
                ("preprocess", clone(unscaled)),
                ("model", DummyRegressor(strategy="mean")),
            ]
        ),
        "LinearRegression_W2": Pipeline(
            [
                ("preprocess", clone(scaled)),
                ("model", LinearRegression()),
            ]
        ),
        "PolynomialRidge_W3": Pipeline(
            [
                (
                    "preprocess",
                    make_polynomial_preprocessor(
                        feature_columns, country_policy
                    ),
                ),
                ("model", Ridge(alpha=1.0)),
            ]
        ),
        "Lasso_W3": Pipeline(
            [
                ("preprocess", clone(scaled)),
                (
                    "model",
                    Lasso(
                        alpha=0.01,
                        max_iter=50_000,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "RandomForest_W5_Extension": Pipeline(
            [
                ("preprocess", clone(unscaled)),
                (
                    "model",
                    RandomForestRegressor(
                        n_estimators=400,
                        min_samples_leaf=2,
                        random_state=RANDOM_STATE,
                        n_jobs=1,
                    ),
                ),
            ]
        ),
        "HistGradientBoosting_BeyondClass": Pipeline(
            [
                ("preprocess", clone(unscaled)),
                (
                    "model",
                    HistGradientBoostingRegressor(
                        learning_rate=0.05,
                        max_iter=300,
                        early_stopping=False,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
    }
    if CASE_COUNT_COL in feature_columns:
        estimators["CaseCountOffset"] = CaseCountOffsetRegressor()
    return estimators

CANDIDATES = make_candidate_estimators(
    FEATURE_COLUMNS, ACTIVE_COUNTRY_POLICY
)
candidate_rationale = pd.DataFrame(
    [
        ["DummyMean", "baseline", "mean-only", "none", "must be beaten"],
        ["CaseCountOffset", "problem baseline", "CaseCount + offset", "one constant", "proxy dependence"],
        ["LinearRegression_W2", "Week 2", "linear", "linear hypothesis", "nonlinearity/collinearity"],
        ["PolynomialRidge_W3", "Week 3", "polynomial + L2", "degree/alpha", "feature expansion"],
        ["Lasso_W3", "Week 3", "linear + L1", "alpha/sparsity", "unstable selection"],
        ["RandomForest_W5_Extension", "Week 5 extension", "averaged trees", "depth/leaf/averaging", "extrapolation/complexity"],
        ["HistGradientBoosting_BeyondClass", "beyond class", "sequential boosting", "rate/leaves/iterations", "tuning sensitivity"],
    ],
    columns=[
        "model",
        "course_link",
        "hypothesis",
        "capacity_control",
        "main_risk",
    ],
)
display(candidate_rationale)


In [ ]:
# ============================================================================
# NOTEBOOK STEP 5.2 — Shared grouped-CV fold manifest
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/08_validation_and_hyperparameter_tuning.md
# Workflow heading: Phase 08 > Core tasks checklist
# Checklist: Match the splitter to generalisation; use the same folds for every candidate.
# Data scope: development IDs/groups only; no fitting.
# Expected output / gate: Per-fold Country/row counts prove entity isolation and fold reuse.
# ============================================================================

grouped_fold_rows = []
for fold_number, (fit_idx, valid_idx) in enumerate(
    GROUP_CV.split(
        X_development,
        y_development,
        groups_development,
    ),
    start=1,
):
    fit_groups = set(groups_development.iloc[fit_idx])
    valid_groups = set(groups_development.iloc[valid_idx])
    assert fit_groups.isdisjoint(valid_groups)
    grouped_fold_rows.append(
        {
            "fold": fold_number,
            "fit_rows": len(fit_idx),
            "validation_rows": len(valid_idx),
            "fit_countries": len(fit_groups),
            "validation_countries": len(valid_groups),
            "country_overlap": len(fit_groups & valid_groups),
        }
    )
grouped_fold_manifest = pd.DataFrame(grouped_fold_rows)
display(grouped_fold_manifest)


In [ ]:
# ============================================================================
# NOTEBOOK STEP 5.3 — Initial-configuration candidate comparison
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/07_model_training.md
# - Machine Learning tổng/machine_learning_lifecycle_phases/09_evaluation_metrics.md
# Workflow heading: Compare training configurations fairly against baseline
# Checklist: Evaluate predeclared non-baseline configurations on identical group folds.
# Data scope: development_df only; gated.
# Expected output / gate: Initial means, spread, gaps and runtime determine which families merit tuning.
# ============================================================================

initial_comparison = None
if RUN_MODEL_COMPARISON:
    validate_primary_metric()
    validate_development_policy()
    comparison_candidates = {
        name: estimator
        for name, estimator in CANDIDATES.items()
        if name not in {"DummyMean", "CaseCountOffset"}
    }
    initial_comparison = compare_models_grouped_cv(
        comparison_candidates,
        X_development,
        y_development,
        groups_development,
        GROUP_CV,
    )
    display(initial_comparison)
else:
    print("Initial candidate comparison skipped (safe default).")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 5.4 — Evidence-based tuning selection
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/08_validation_and_hyperparameter_tuning.md
# Workflow heading: Phase 08 > Search strategy and exit criteria
# Checklist: Select a small subset from initial CV evidence; do not tune every model.
# Data scope: Configuration only.
# Expected output / gate: SELECTED_TUNING_MODELS is empty until the written Phase 5.3 decision is complete.
# ============================================================================

SELECTED_TUNING_MODELS = []  # TODO: populate from Step 5.3 evidence
TUNING_SPECS = {}
searches = {}
tuning_summary_rows = []
print("Selected tuning candidates:", SELECTED_TUNING_MODELS)


In [ ]:
# ============================================================================
# NOTEBOOK STEP 5.5 — Week 3 Ridge/Lasso search spaces
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/08_validation_and_hyperparameter_tuning.md
# Workflow heading: Search APIs, configuration keywords and sensible ranges
# Checklist: Jointly test polynomial degree/alpha; test L1 strength over a log scale.
# Data scope: Configuration only; no fitting in this cell.
# Expected output / gate: Search spaces have an explicit capacity/regularisation rationale.
# ============================================================================

TUNING_SPECS["PolynomialRidge_W3"] = {
    "estimator": CANDIDATES["PolynomialRidge_W3"],
    "param_grid": {
        "preprocess__numeric_poly__poly__degree": [1, 2],
        "model__alpha": np.logspace(-4, 4, 9),
    },
}
TUNING_SPECS["Lasso_W3"] = {
    "estimator": CANDIDATES["Lasso_W3"],
    "param_grid": {
        "model__alpha": np.logspace(-5, 1, 13),
    },
}
print("Week 3 search spaces:", list(TUNING_SPECS))


In [ ]:
# ============================================================================
# NOTEBOOK STEP 5.6 — Week 5 Random Forest regression-extension search space
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/08_validation_and_hyperparameter_tuning.md
# Workflow heading: Common hyperparameter keywords and sensible search ranges
# Checklist: Control tree depth, leaf size and feature subsampling.
# Data scope: Configuration only; no fitting in this cell.
# Expected output / gate: Forest capacity/averaging choices are explicit and reproducible.
# ============================================================================

TUNING_SPECS["RandomForest_W5_Extension"] = {
    "estimator": CANDIDATES["RandomForest_W5_Extension"],
    "param_grid": {
        "model__max_depth": [None, 6, 12],
        "model__min_samples_leaf": [1, 2, 5, 10],
        "model__max_features": [0.5, 0.8, 1.0],
    },
}


In [ ]:
# ============================================================================
# NOTEBOOK STEP 5.7A — Beyond-class HistGradientBoosting search space
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/08_validation_and_hyperparameter_tuning.md
# Workflow heading: Common hyperparameter keywords and sensible search ranges
# Checklist: Control shrinkage, leaf capacity, minimum leaf size and L2 regularisation.
# Data scope: Configuration only; no fitting in this cell.
# Expected output / gate: Beyond-class search is bounded and tied to the nonlinear EDA hypothesis.
# ============================================================================

TUNING_SPECS["HistGradientBoosting_BeyondClass"] = {
    "estimator": CANDIDATES["HistGradientBoosting_BeyondClass"],
    "param_grid": {
        "model__learning_rate": [0.03, 0.05, 0.10],
        "model__max_leaf_nodes": [7, 15, 31],
        "model__min_samples_leaf": [10, 20, 40],
        "model__l2_regularization": [0.0, 0.1, 1.0],
    },
}
print("All available tuning specifications:", list(TUNING_SPECS))


In [ ]:
# ============================================================================
# NOTEBOOK STEP 5.7B — Execute selected grouped GridSearchCV runs
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/08_validation_and_hyperparameter_tuning.md
# Workflow heading: Evaluation helpers, Search APIs and search configuration keywords
# Checklist: Keep preprocessing inside search; record validation/train metrics, spread and time.
# Data scope: development_df only; expensive and gated.
# Expected output / gate: Only selected families run; search objects and comparable summaries are retained.
# ============================================================================

def run_grouped_search(model_name: str):
    spec = TUNING_SPECS[model_name]
    primary_key = validate_primary_metric()
    started = time.perf_counter()
    search = GridSearchCV(
        estimator=spec["estimator"],
        param_grid=spec["param_grid"],
        scoring=SCORING,
        cv=GROUP_CV,
        n_jobs=N_JOBS,
        return_train_score=True,
        refit=primary_key,
        error_score="raise",
    )
    search.fit(
        X_development,
        y_development,
        groups=groups_development,
    )
    best = search.best_index_
    primary_valid = scorer_value_for_display(
        search.cv_results_[f"mean_test_{primary_key}"][best],
        primary_key,
    )
    primary_train = scorer_value_for_display(
        search.cv_results_[f"mean_train_{primary_key}"][best],
        primary_key,
    )
    primary_gap = (
        primary_valid - primary_train
        if primary_key in LOSS_METRICS
        else primary_train - primary_valid
    )
    row = {
        "model": model_name,
        "primary_metric": primary_key,
        "primary_cv_mean": primary_valid,
        "primary_cv_std": search.cv_results_[
            f"std_test_{primary_key}"
        ][best],
        "primary_train_mean": primary_train,
        "primary_gap": primary_gap,
        "cv_rmse_mean": -search.cv_results_["mean_test_rmse"][best],
        "cv_rmse_std": search.cv_results_["std_test_rmse"][best],
        "cv_mae_mean": -search.cv_results_["mean_test_mae"][best],
        "cv_mae_std": search.cv_results_["std_test_mae"][best],
        "cv_r2_mean": search.cv_results_["mean_test_r2"][best],
        "cv_r2_std": search.cv_results_["std_test_r2"][best],
        "best_params": search.best_params_,
        "fit_time_mean": search.cv_results_["mean_fit_time"][best],
        "score_time_mean": search.cv_results_["mean_score_time"][best],
        "runtime_seconds": time.perf_counter() - started,
    }
    return search, row

if RUN_TUNING:
    assert RUN_MODEL_COMPARISON
    assert SELECTED_TUNING_MODELS
    assert set(SELECTED_TUNING_MODELS).issubset(TUNING_SPECS)
    validate_development_policy()
    for selected_name in SELECTED_TUNING_MODELS:
        selected_search, selected_row = run_grouped_search(selected_name)
        searches[selected_name] = selected_search
        tuning_summary_rows.append(selected_row)
    tuning_summary = pd.DataFrame(tuning_summary_rows).sort_values(
        "primary_cv_mean",
        ascending=PRIMARY_METRIC_KEY in LOSS_METRICS,
    )
    display(tuning_summary)
else:
    tuning_summary = pd.DataFrame()
    print("Tuning skipped (safe default).")


### Training note — complete once per expensive run

- Objective / hypothesis: TODO
- Data scope: development only
- Pipeline and ordered features: TODO
- Group splitter/folds: `Country`, TODO folds
- Primary/secondary metrics: TODO
- Search space rationale: TODO
- Seed / `n_jobs` / environment: TODO
- Runtime, warnings and failed fits: TODO
- Best parameters and transformed feature count: TODO
- Fold stability and train–validation gap: TODO
- Decision / next action: TODO

For Ridge/Lasso, also record degree, alpha candidates, scaling, coefficient
shrinkage and number of zero Lasso coefficients.


In [ ]:
# ============================================================================
# NOTEBOOK STEP 5.8 — Programmatic consolidated comparison table
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/09_evaluation_metrics.md
# Workflow heading: Phase 09 > Core tasks checklist
# Checklist: Report primary/secondary metrics, variation and fair baseline comparison.
# Data scope: Previously computed development-CV results only.
# Expected output / gate: One table joins baseline, initial and tuned evidence without manual retyping.
# ============================================================================

comparison_parts = []
if baseline_results is not None:
    baseline_part = baseline_results.copy()
    baseline_part["stage"] = "baseline"
    comparison_parts.append(baseline_part)
if initial_comparison is not None:
    initial_part = initial_comparison.copy()
    initial_part["stage"] = "initial"
    comparison_parts.append(initial_part)
if not tuning_summary.empty:
    tuned_part = tuning_summary.copy()
    tuned_part["stage"] = "tuned"
    comparison_parts.append(tuned_part)

if comparison_parts:
    consolidated_comparison = (
        pd.concat(comparison_parts, ignore_index=True, sort=False)
        .sort_values(
            "primary_cv_mean",
            ascending=PRIMARY_METRIC_KEY in LOSS_METRICS,
        )
        .reset_index(drop=True)
    )
    display(consolidated_comparison)
else:
    consolidated_comparison = pd.DataFrame()
    print("No CV result tables exist yet; complete Steps 4.7 and 5.3–5.7.")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 5.9 — Best-search fold stability table
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/08_validation_and_hyperparameter_tuning.md
# Workflow heading: Phase 08 > Record fold results and train–validation gaps
# Checklist: Inspect each selected configuration by fold, not mean alone.
# Data scope: Previously fitted development searches only.
# Expected output / gate: Fold-level primary values expose instability hidden by an aggregate mean.
# ============================================================================

tuning_fold_rows = []
for model_name, search in searches.items():
    best = search.best_index_
    for fold_index in range(N_GROUP_FOLDS):
        raw_value = search.cv_results_[
            f"split{fold_index}_test_{PRIMARY_METRIC_KEY}"
        ][best]
        tuning_fold_rows.append(
            {
                "model": model_name,
                "fold": fold_index + 1,
                "primary_metric": PRIMARY_METRIC_KEY,
                "validation_value": scorer_value_for_display(
                    raw_value, PRIMARY_METRIC_KEY
                ),
            }
        )
tuning_fold_stability = pd.DataFrame(tuning_fold_rows)
if not tuning_fold_stability.empty:
    display(tuning_fold_stability)
    display(
        tuning_fold_stability.groupby("model")["validation_value"]
        .agg(["mean", "std", "min", "max"])
    )
else:
    print("No tuned fold results exist yet.")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 5.10A — Executable finalist/model specification utilities
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/08_validation_and_hyperparameter_tuning.md
# - Machine Learning tổng/machine_learning_lifecycle_phases/11_final_evaluation.md
# - Machine Learning tổng/machine_learning_lifecycle_phases/12_model_finalisation_and_packaging.md
# Workflow heading: CV-only selection / freeze before final evaluation / versioned packaging
# Checklist: Freeze candidate, params, ordered features and data policies without an in-memory search dependency.
# Data scope: Definitions only.
# Expected output / gate: Any finalist can be rebuilt after a kernel restart and validated against known parameters.
# ============================================================================

REQUIRED_SPEC_KEYS = {
    "candidate",
    "params",
    "features",
    "status_invalid_policy",
    "country_feature_policy",
}

def validate_model_spec(spec: dict) -> None:
    missing_keys = REQUIRED_SPEC_KEYS - set(spec)
    assert not missing_keys, (
        f"Model spec is missing key(s): {sorted(missing_keys)}"
    )
    assert isinstance(spec["features"], list) and spec["features"]
    assert len(spec["features"]) == len(set(spec["features"]))
    assert set(spec["features"]).issubset(set(ALL_CANDIDATE_FEATURES))
    assert spec["status_invalid_policy"] in {
        "retain_numeric",
        "invalid_to_nan",
    }
    assert spec["country_feature_policy"] in {"group_only", "one_hot"}
    if spec["country_feature_policy"] == "group_only":
        assert GROUP_COL not in spec["features"]
    else:
        assert GROUP_COL in spec["features"]
    if spec["candidate"] == "CaseCountOffset":
        assert spec["features"] == [CASE_COUNT_COL]
        assert spec["country_feature_policy"] == "group_only"
        assert (
            spec["params"].get("case_count_col", CASE_COUNT_COL)
            == CASE_COUNT_COL
        )
    available = make_candidate_estimators(
        spec["features"], spec["country_feature_policy"]
    )
    assert spec["candidate"] in available
    unknown_params = set(spec["params"]) - set(
        available[spec["candidate"]].get_params(deep=True)
    )
    assert not unknown_params, (
        f"Unknown model parameter(s): {sorted(unknown_params)}"
    )

def build_model_from_spec(spec: dict):
    validate_model_spec(spec)
    estimator = make_candidate_estimators(
        spec["features"], spec["country_feature_policy"]
    )[spec["candidate"]]
    return estimator.set_params(**spec["params"])

def prepare_X_from_spec(
    frame: pd.DataFrame,
    spec: dict,
) -> pd.DataFrame:
    validate_model_spec(spec)
    return prepare_feature_frame(
        frame,
        spec["features"],
        spec["status_invalid_policy"],
    )

def stable_spec_signature(spec: dict) -> str:
    validate_model_spec(spec)
    payload = {
        "model_spec": spec,
        "random_state": RANDOM_STATE,
        "group_folds": N_GROUP_FOLDS,
        "holdout_fraction": INTERNAL_HOLDOUT_FRACTION,
        "target": TARGET_COL,
        "primary_metric_key": PRIMARY_METRIC_KEY,
        "final_cv_primary_reference": FINAL_CV_PRIMARY_REFERENCE,
        "holdout_rule_confirmed": HOLDOUT_ACCEPTANCE_RULE_CONFIRMED,
        "holdout_max_relative_primary_degradation":
            HOLDOUT_MAX_RELATIVE_PRIMARY_DEGRADATION,
        "holdout_max_absolute_bias": HOLDOUT_MAX_ABSOLUTE_BIAS,
        "train_sha256": hashlib.sha256(TRAIN_PATH.read_bytes()).hexdigest(),
        "test_sha256": hashlib.sha256(TEST_PATH.read_bytes()).hexdigest(),
        "development_record_ids": development_df[ID_COL].tolist(),
        "holdout_record_ids": holdout_df[ID_COL].tolist(),
    }
    encoded = json.dumps(
        payload, sort_keys=True, default=str
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()

def build_frozen_finalist():
    assert FINAL_SPEC_CONFIRMED
    return build_model_from_spec(FINAL_MODEL_SPEC)


In [ ]:
# ============================================================================
# NOTEBOOK STEP 5.10B — Finalist freeze validation
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/08_validation_and_hyperparameter_tuning.md
# - Machine Learning tổng/machine_learning_lifecycle_phases/11_final_evaluation.md
# Workflow heading: Select using CV only; freeze before holdout
# Checklist: Require at least two finalists for diagnostic comparison.
# Data scope: Configuration validation only; no fitting.
# Expected output / gate: Every populated finalist is rebuildable and distinct.
# ============================================================================

if FINALIST_SPECS:
    assert len(FINALIST_SPECS) >= 2
    for finalist_label, finalist_spec in FINALIST_SPECS.items():
        validate_model_spec(finalist_spec)
        print(
            finalist_label,
            finalist_spec["candidate"],
            stable_spec_signature(finalist_spec)[:16],
        )
else:
    print(
        "FINALIST_SPECS is empty. Populate it only after consolidated "
        "development-CV evidence is interpreted."
    )


### Phase 5 decision gate

- [ ] Search spaces have a capacity/regularisation rationale.
- [ ] Means, fold spread, gaps, secondary metrics and runtime are reported.
- [ ] No holdout or external-value result influenced training/tuning.
- [ ] At least two finalists are frozen as literal reconstructable specs.

**Decision:** TODO — name the finalists and the diagnostic that could still
disqualify each one.


# PHASE 6 — Error Analysis, Robustness, Interpretation & Ultimate Judgment

**Lifecycle coverage:** `10_error_analysis_and_robustness.md`

> **Workflow adaptation:** cross_val_predict outputs are diagnostic after model selection; pooled OOF scores are not claimed as an independent unbiased final estimate.

| Notebook step | Concrete purpose | Exact workflow file |
|---|---|---|
| 6.1 | Generate post-selection grouped OOF predictions | `10_error_analysis_and_robustness.md` |
| 6.2 | Plot actual versus predicted | `10_error_analysis_and_robustness.md` |
| 6.3 | Analyse residual patterns | `10_error_analysis_and_robustness.md` |
| 6.4 | Inspect worst-error records | `10_error_analysis_and_robustness.md` |
| 6.5 | Compare meaningful error slices | `10_error_analysis_and_robustness.md` |
| 6.6 | Run prediction-sensitivity stress tests | `10_error_analysis_and_robustness.md` |
| 6.7 | Run finalist feature/rule ablations | `10_error_analysis_and_robustness.md` |
| 6.8 | Interpret models using family-appropriate evidence | `10_error_analysis_and_robustness.md` |
| 6.9 | Complete a decision matrix | `10_error_analysis_and_robustness.md` |
| 6.10 | Freeze the ultimate judgment/specification | `11_final_evaluation.md` |

Every code cell below repeats its workflow source, data scope, and expected
output/gate in comments. Heavy training, holdout access, and file export remain
behind explicit flags.


In [ ]:
# ============================================================================
# NOTEBOOK STEP 6.1 — Grouped OOF predictions for multiple finalists
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/10_error_analysis_and_robustness.md
# Workflow heading: Exact diagnostic APIs > Error tables / cross_val_predict
# Checklist: Generate one prediction per development row from a fold that excluded its Country.
# Data scope: development_df only; expensive and gated.
# Expected output / gate: OOF tables/summary exist for at least two frozen finalists.
# ============================================================================

oof_results_by_finalist = {}
oof_summary = None
if RUN_OOF_ANALYSIS:
    assert len(FINALIST_SPECS) >= 2
    assert OOF_DIAGNOSTIC_LABEL in FINALIST_SPECS
    oof_summary_rows = []
    for finalist_label, finalist_spec in FINALIST_SPECS.items():
        finalist_model = build_model_from_spec(finalist_spec)
        finalist_X = prepare_X_from_spec(
            development_df, finalist_spec
        )
        predictions = cross_val_predict(
            finalist_model,
            finalist_X,
            y_development,
            groups=groups_development,
            cv=GROUP_CV,
            n_jobs=N_JOBS,
            method="predict",
        )
        results = development_df[
            [ID_COL, GROUP_COL, "Year", CASE_COUNT_COL, TARGET_COL]
        ].copy()
        results["prediction"] = predictions
        results["residual"] = (
            results[TARGET_COL] - results["prediction"]
        )
        results["absolute_error"] = results["residual"].abs()
        results["had_missing_selected_feature"] = (
            finalist_X.isna().any(axis=1).to_numpy()
        )
        oof_results_by_finalist[finalist_label] = results
        oof_summary_rows.append(
            {
                "finalist": finalist_label,
                "candidate": finalist_spec["candidate"],
                "pooled_oof_rmse": root_mean_squared_error(
                    y_development, predictions
                ),
                "pooled_oof_mae": mean_absolute_error(
                    y_development, predictions
                ),
                "pooled_oof_r2": r2_score(
                    y_development, predictions
                ),
                "mean_residual": float(
                    (y_development.to_numpy() - predictions).mean()
                ),
                "feature_count": len(finalist_spec["features"]),
            }
        )
    oof_summary = pd.DataFrame(oof_summary_rows)
    display(oof_summary)
    print("Post-selection diagnostic evidence only; holdout remains closed.")
else:
    print("OOF analysis skipped (safe default).")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 6.2 — OOF actual-versus-predicted plot
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/10_error_analysis_and_robustness.md
# Workflow heading: Worst cases and residual diagnostics
# Checklist: Check calibration, compression and systematic under/over-prediction.
# Data scope: Selected OOF diagnostic table only.
# Expected output / gate: Plot is available only after OOF generation.
# ============================================================================

if OOF_DIAGNOSTIC_LABEL in oof_results_by_finalist:
    oof_results = oof_results_by_finalist[
        OOF_DIAGNOSTIC_LABEL
    ].copy()
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(
        oof_results[TARGET_COL],
        oof_results["prediction"],
        alpha=0.45,
        s=18,
    )
    lower = min(
        oof_results[TARGET_COL].min(),
        oof_results["prediction"].min(),
    )
    upper = max(
        oof_results[TARGET_COL].max(),
        oof_results["prediction"].max(),
    )
    ax.plot([lower, upper], [lower, upper], "--", color="tab:red")
    ax.set(
        title="OOF actual vs predicted",
        xlabel="Actual",
        ylabel="Predicted",
    )
    plt.tight_layout()
    plt.show()
else:
    print("No selected OOF result is available.")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 6.3 — OOF residual-pattern plots
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/10_error_analysis_and_robustness.md
# Workflow heading: Residuals by prediction, time and important predictors
# Checklist: Look for bias, heteroscedasticity, nonlinear structure and proxy dependence.
# Data scope: Selected OOF diagnostic table only.
# Expected output / gate: Residual plots become hypotheses; they are not automatically proven causes.
# ============================================================================

if OOF_DIAGNOSTIC_LABEL in oof_results_by_finalist:
    oof_results = oof_results_by_finalist[
        OOF_DIAGNOSTIC_LABEL
    ].copy()
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    axes[0].scatter(
        oof_results["prediction"],
        oof_results["residual"],
        alpha=0.45,
        s=18,
    )
    axes[0].set(
        title="Residual vs prediction",
        xlabel="Prediction",
        ylabel="Residual",
    )
    axes[1].scatter(
        oof_results[CASE_COUNT_COL],
        oof_results["residual"],
        alpha=0.45,
        s=18,
    )
    axes[1].set(
        title="Residual vs CaseCount",
        xlabel=CASE_COUNT_COL,
        ylabel="Residual",
    )
    axes[2].scatter(
        oof_results["Year"],
        oof_results["residual"],
        alpha=0.45,
        s=18,
    )
    axes[2].set(
        title="Residual vs Year",
        xlabel="Year",
        ylabel="Residual",
    )
    for axis in axes:
        axis.axhline(0, linestyle="--", color="tab:red")
    plt.tight_layout()
    plt.show()
else:
    print("No selected OOF result is available.")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 6.4 — Worst OOF error records
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/10_error_analysis_and_robustness.md
# Workflow heading: Inspect the worst or most costly mistakes
# Checklist: Identify record/group/target regions driving the largest errors.
# Data scope: Selected OOF diagnostic table only.
# Expected output / gate: Worst rows are displayed with explanatory fields for manual investigation.
# ============================================================================

worst_oof_errors = None
if OOF_DIAGNOSTIC_LABEL in oof_results_by_finalist:
    oof_results = oof_results_by_finalist[
        OOF_DIAGNOSTIC_LABEL
    ].copy()
    worst_oof_errors = oof_results.nlargest(
        15, "absolute_error"
    )
    display(worst_oof_errors)
else:
    print("No selected OOF result is available.")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 6.5 — Country, target, year, CaseCount and missingness error slices
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/10_error_analysis_and_robustness.md
# Workflow heading: Slice errors by time, target range, missingness, rarity and group
# Checklist: Compare MAE, bias and RMSE across predeclared meaningful slices.
# Data scope: Selected OOF diagnostic table only.
# Expected output / gate: Slice tables identify concentrated failure modes rather than only global means.
# ============================================================================

def error_slice_table(
    results: pd.DataFrame,
    slice_column: str,
) -> pd.DataFrame:
    return (
        results.groupby(
            slice_column,
            dropna=False,
            observed=True,
        )
        .agg(
            rows=(TARGET_COL, "size"),
            mae=("absolute_error", "mean"),
            bias=("residual", "mean"),
            rmse=(
                "residual",
                lambda values: float(
                    np.sqrt(np.mean(np.square(values)))
                ),
            ),
        )
        .sort_values("mae", ascending=False)
    )

oof_slice_tables = {}
if OOF_DIAGNOSTIC_LABEL in oof_results_by_finalist:
    sliced_oof = oof_results_by_finalist[
        OOF_DIAGNOSTIC_LABEL
    ].copy()
    sliced_oof["target_quantile"] = pd.qcut(
        sliced_oof[TARGET_COL], q=4, duplicates="drop"
    )
    sliced_oof["year_band"] = pd.qcut(
        sliced_oof["Year"], q=4, duplicates="drop"
    )
    sliced_oof["case_count_band"] = pd.qcut(
        sliced_oof[CASE_COUNT_COL], q=4, duplicates="drop"
    )
    for slice_name in [
        GROUP_COL,
        "target_quantile",
        "year_band",
        "case_count_band",
        "had_missing_selected_feature",
    ]:
        oof_slice_tables[slice_name] = error_slice_table(
            sliced_oof, slice_name
        )
        print(f"\nError slice: {slice_name}")
        display(oof_slice_tables[slice_name].head(20))
else:
    print("No selected OOF result is available.")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 6.6 — Prediction-sensitivity robustness stress tests
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/10_error_analysis_and_robustness.md
# Workflow heading: Stress tests for missing, extreme, shifted and unseen inputs
# Checklist: Measure prediction change under predeclared plausible perturbations.
# Data scope: Development feature sample only; labels are not used for stress scoring; gated.
# Expected output / gate: Finite-rate and prediction-delta summaries expose fragile model behaviour.
# ============================================================================

robustness_summary = None
if RUN_ROBUSTNESS_TESTS:
    assert FINALIST_SPECS
    robustness_rows = []
    for finalist_label, finalist_spec in FINALIST_SPECS.items():
        model = build_model_from_spec(finalist_spec)
        finalist_X = prepare_X_from_spec(
            development_df, finalist_spec
        )
        model.fit(finalist_X, y_development)
        probe = finalist_X.head(min(200, len(finalist_X))).copy()
        reference_prediction = model.predict(probe)
        scenarios = {}
        numeric_features = [
            col
            for col in finalist_spec["features"]
            if col != GROUP_COL
        ]
        if numeric_features:
            stressed_feature = numeric_features[0]
            missing_probe = probe.copy()
            missing_probe[stressed_feature] = np.nan
            scenarios[f"missing_{stressed_feature}"] = missing_probe

            high_probe = probe.copy()
            observed = finalist_X[stressed_feature]
            observed_range = observed.max() - observed.min()
            high_probe[stressed_feature] = (
                observed.max() + max(observed_range, 1)
            )
            scenarios[f"high_{stressed_feature}"] = high_probe
        if GROUP_COL in finalist_spec["features"]:
            unseen_probe = probe.copy()
            unseen_probe[GROUP_COL] = (
                pd.to_numeric(finalist_X[GROUP_COL], errors="coerce").max()
                + 1
            )
            scenarios["unseen_Country"] = unseen_probe
        if CASE_COUNT_COL in finalist_spec["features"]:
            shifted_probe = probe.copy()
            shifted_probe[CASE_COUNT_COL] = (
                shifted_probe[CASE_COUNT_COL]
                + development_df[CASE_COUNT_COL].std()
            )
            scenarios["CaseCount_plus_1sd"] = shifted_probe

        for scenario_name, scenario_X in scenarios.items():
            scenario_prediction = model.predict(scenario_X)
            delta = scenario_prediction - reference_prediction
            robustness_rows.append(
                {
                    "finalist": finalist_label,
                    "scenario": scenario_name,
                    "finite_prediction_rate": float(
                        np.isfinite(scenario_prediction).mean()
                    ),
                    "mean_absolute_prediction_change": float(
                        np.mean(np.abs(delta))
                    ),
                    "max_absolute_prediction_change": float(
                        np.max(np.abs(delta))
                    ),
                }
            )
    robustness_summary = pd.DataFrame(robustness_rows)
    display(robustness_summary)
else:
    print("Robustness stress tests skipped (safe default).")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 6.7 — Finalist CaseCount ablation under fixed grouped CV
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/10_error_analysis_and_robustness.md
# Workflow heading: Feature/data-rule ablations and leakage audit
# Checklist: Compare each eligible finalist with and without CaseCount under identical folds.
# Data scope: development_df only; expensive and gated.
# Expected output / gate: Ablation performance quantifies shortcut dependence without touching holdout.
# ============================================================================

finalist_ablation_summary = None
if RUN_FINALIST_ABLATIONS:
    assert FINALIST_SPECS
    ablation_rows = []
    for finalist_label, full_spec in FINALIST_SPECS.items():
        if (
            CASE_COUNT_COL not in full_spec["features"]
            or full_spec["candidate"] == "CaseCountOffset"
        ):
            continue
        without_case_spec = {
            **full_spec,
            "features": [
                col
                for col in full_spec["features"]
                if col != CASE_COUNT_COL
            ],
        }
        for variant, variant_spec in {
            "with_CaseCount": full_spec,
            "without_CaseCount": without_case_spec,
        }.items():
            variant_X = prepare_X_from_spec(
                development_df, variant_spec
            )
            variant_model = build_model_from_spec(variant_spec)
            row = compare_models_grouped_cv(
                {f"{finalist_label}:{variant}": variant_model},
                variant_X,
                y_development,
                groups_development,
                GROUP_CV,
            ).iloc[0].to_dict()
            row.update(
                {
                    "finalist": finalist_label,
                    "variant": variant,
                }
            )
            ablation_rows.append(row)
    finalist_ablation_summary = pd.DataFrame(ablation_rows)
    display(finalist_ablation_summary)
else:
    print("Finalist ablations skipped (safe default).")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 6.8 — Model-family-appropriate interpretation
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/10_error_analysis_and_robustness.md
# Workflow heading: Interpretation and hypotheses versus proven causes
# Checklist: Inspect coefficients or tree importances only where the estimator exposes them.
# Data scope: development_df only; fitted finalist models; gated.
# Expected output / gate: Interpretation tables are labelled associational, not causal.
# ============================================================================

interpretation_tables = {}
if RUN_MODEL_INTERPRETATION:
    assert FINALIST_SPECS
    for finalist_label, finalist_spec in FINALIST_SPECS.items():
        model = build_model_from_spec(finalist_spec)
        finalist_X = prepare_X_from_spec(
            development_df, finalist_spec
        )
        model.fit(finalist_X, y_development)
        if isinstance(model, Pipeline):
            fitted_preprocessor = model.named_steps["preprocess"]
            fitted_estimator = model.named_steps["model"]
            feature_names = fitted_preprocessor.get_feature_names_out()
            if hasattr(fitted_estimator, "coef_"):
                values = np.ravel(fitted_estimator.coef_)
                value_name = "coefficient"
            elif hasattr(fitted_estimator, "feature_importances_"):
                values = fitted_estimator.feature_importances_
                value_name = "feature_importance"
            else:
                print(
                    finalist_label,
                    "does not expose native coefficients/importances; "
                    "use a predeclared permutation method if needed.",
                )
                continue
            table = (
                pd.DataFrame(
                    {
                        "feature": feature_names,
                        value_name: values,
                    }
                )
                .assign(
                    absolute_value=lambda frame: frame[value_name].abs()
                )
                .sort_values("absolute_value", ascending=False)
            )
            interpretation_tables[finalist_label] = table
            print(f"\n{finalist_label} — associational interpretation")
            display(table.head(30))
        elif isinstance(model, CaseCountOffsetRegressor):
            interpretation_tables[finalist_label] = pd.DataFrame(
                {
                    "feature": [CASE_COUNT_COL, "intercept_offset"],
                    "coefficient_or_value": [1.0, model.offset_],
                }
            )
            display(interpretation_tables[finalist_label])
else:
    print("Model interpretation skipped (safe default).")


## 6.9 Decision matrix

| Criterion | Priority | Finalist A | Finalist B | Judgment |
|---|---|---|---|---|
| Configured primary CV metric | Primary | TODO | TODO | TODO |
| Fold stability | High | TODO | TODO | TODO |
| Train–validation gap | High | TODO | TODO | TODO |
| Worst/slice errors | High | TODO | TODO | TODO |
| CaseCount ablation | High | TODO | TODO | TODO |
| Stress-test sensitivity | High | TODO | TODO | TODO |
| Interpretation | Medium | TODO | TODO | TODO |
| Runtime/reproducibility | Medium | TODO | TODO | TODO |


In [ ]:
# ============================================================================
# NOTEBOOK STEP 6.10 — Ultimate model-specification freeze
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/11_final_evaluation.md
# - Machine Learning tổng/machine_learning_lifecycle_phases/12_model_finalisation_and_packaging.md
# Workflow heading: Freeze before final evaluation / version the final artefact
# Checklist: Confirm the final spec is one diagnosed finalist and compute its signature.
# Data scope: Configuration and hashes only; no fitting or holdout access.
# Expected output / gate: A stable final signature exists before RUN_FINAL_HOLDOUT can be enabled.
# ============================================================================

frozen_final_spec_signature = None
if FINAL_SPEC_CONFIRMED:
    validate_model_spec(FINAL_MODEL_SPEC)
    assert FINALIST_SPECS
    finalist_signatures = {
        stable_spec_signature(spec)
        for spec in FINALIST_SPECS.values()
    }
    frozen_final_spec_signature = stable_spec_signature(
        FINAL_MODEL_SPEC
    )
    assert frozen_final_spec_signature in finalist_signatures
    print("Frozen final signature:", frozen_final_spec_signature)
else:
    print("Final specification remains unconfirmed; holdout stays closed.")


## 6.10 Ultimate judgment — writing scaffold

**Claim:** TODO — name the selected model and supported offline setting.  
**Evidence:** TODO — cite baseline, grouped CV mean ± SD, gap, OOF errors,
ablation and robustness evidence.  
**Reasoning:** TODO — explain why evidence outweighs alternatives.  
**Limitations:** TODO — unseen countries, CaseCount shift/proxy risk, anomalous
values, target support and finite-group uncertainty.  
**Boundary:** TODO — state what the experiment cannot prove.

### Phase 6 decision gate

- [ ] Model is selected from development CV/OOF evidence only.
- [ ] At least one strong alternative is compared fairly.
- [ ] Failure modes and evidence boundaries are explicit.
- [ ] Model, features, policies, parameters and acceptance rule are frozen.


# PHASE 7 — One-Time Holdout, Final Refit & Submission

**Lifecycle coverage:** `11_final_evaluation.md` + `12_model_finalisation_and_packaging.md`

> **Workflow adaptation:** The locked labelled holdout is final evaluation. The supplied test.csv has no target and is final inference/submission data, not evaluation.

| Notebook step | Concrete purpose | Exact workflow file |
|---|---|---|
| 7.1 | Fit frozen spec and evaluate locked labelled holdout once | `11_final_evaluation.md` |
| 7.2 | Apply predeclared acceptance rule | `11_final_evaluation.md` |
| 7.3 | Refit selected pipeline on all labelled rows | `12_model_finalisation_and_packaging.md` |
| 7.4 | Audit post-freeze external compatibility | `12_model_finalisation_and_packaging.md` |
| 7.5 | Generate continuous external predictions | `12_model_finalisation_and_packaging.md` |
| 7.6 | Validate submission contract | `12_model_finalisation_and_packaging.md` |
| 7.7 | Write and round-trip exact CSV | `12_model_finalisation_and_packaging.md` |

Every code cell below repeats its workflow source, data scope, and expected
output/gate in comments. Heavy training, holdout access, and file export remain
behind explicit flags.


In [ ]:
# ============================================================================
# NOTEBOOK STEP 7.1 — One-time locked labelled-holdout evaluation
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/11_final_evaluation.md
# Workflow heading: Phase 11 > Core tasks checklist
# Checklist: Freeze first; fit on development only; predict holdout without refitting on it.
# Data scope: development features/labels + one-time holdout features/labels; gated.
# Expected output / gate: Predefined RMSE/MAE/R², bias and final-spec signature are recorded once.
# ============================================================================

holdout_metrics = None
holdout_predictions = None
holdout_spec_signature = None
fitted_holdout_model = None

if RUN_FINAL_HOLDOUT:
    assert oof_results_by_finalist
    assert FINAL_SPEC_CONFIRMED
    assert frozen_final_spec_signature is not None
    validate_primary_metric()
    assert HOLDOUT_ACCEPTANCE_RULE_CONFIRMED
    assert FINAL_CV_PRIMARY_REFERENCE is not None
    assert HOLDOUT_MAX_RELATIVE_PRIMARY_DEGRADATION is not None
    assert HOLDOUT_MAX_ABSOLUTE_BIAS is not None

    X_final_development = prepare_X_from_spec(
        development_df, FINAL_MODEL_SPEC
    )
    X_holdout = prepare_X_from_spec(
        holdout_df, FINAL_MODEL_SPEC
    )
    y_holdout = holdout_df[TARGET_COL]
    fitted_holdout_model = build_frozen_finalist()
    fitted_holdout_model.fit(
        X_final_development, y_development
    )
    holdout_predictions = fitted_holdout_model.predict(X_holdout)
    holdout_spec_signature = stable_spec_signature(
        FINAL_MODEL_SPEC
    )
    assert holdout_spec_signature == frozen_final_spec_signature
    holdout_metrics = pd.Series(
        {
            "rmse": root_mean_squared_error(
                y_holdout, holdout_predictions
            ),
            "mae": mean_absolute_error(
                y_holdout, holdout_predictions
            ),
            "r2": r2_score(y_holdout, holdout_predictions),
            "mean_residual": float(
                (y_holdout - holdout_predictions).mean()
            ),
        },
        name="One-time Country holdout",
    )
    display(holdout_metrics)
else:
    print("Locked holdout remains unopened (safe default).")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 7.2 — Predeclared holdout acceptance decision
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/11_final_evaluation.md
# Workflow heading: Phase 11 > Acceptance criteria and exit criteria
# Checklist: Compare holdout primary/bias against thresholds frozen before access.
# Data scope: Previously computed one-time holdout metrics only.
# Expected output / gate: Pass/fail and relative degradation are explicit; failure is reported, not retuned.
# ============================================================================

holdout_acceptance_passed = None
holdout_decision_evidence = None
if holdout_metrics is not None:
    holdout_primary = holdout_metrics[PRIMARY_METRIC_KEY]
    if PRIMARY_METRIC_KEY in LOSS_METRICS:
        relative_primary_degradation = (
            holdout_primary - FINAL_CV_PRIMARY_REFERENCE
        ) / max(
            abs(FINAL_CV_PRIMARY_REFERENCE),
            np.finfo(float).eps,
        )
    else:
        relative_primary_degradation = (
            FINAL_CV_PRIMARY_REFERENCE - holdout_primary
        ) / max(
            abs(FINAL_CV_PRIMARY_REFERENCE),
            np.finfo(float).eps,
        )
    holdout_acceptance_passed = bool(
        relative_primary_degradation
        <= HOLDOUT_MAX_RELATIVE_PRIMARY_DEGRADATION
        and abs(holdout_metrics["mean_residual"])
        <= HOLDOUT_MAX_ABSOLUTE_BIAS
    )
    holdout_decision_evidence = pd.Series(
        {
            "primary_metric": PRIMARY_METRIC_KEY,
            "cv_reference": FINAL_CV_PRIMARY_REFERENCE,
            "holdout_primary": holdout_primary,
            "relative_primary_degradation":
                relative_primary_degradation,
            "max_allowed_relative_degradation":
                HOLDOUT_MAX_RELATIVE_PRIMARY_DEGRADATION,
            "absolute_bias": abs(
                holdout_metrics["mean_residual"]
            ),
            "max_allowed_absolute_bias":
                HOLDOUT_MAX_ABSOLUTE_BIAS,
            "acceptance_passed": holdout_acceptance_passed,
        },
        name="Predeclared holdout decision",
    )
    display(holdout_decision_evidence)
else:
    print("No holdout metrics exist; no acceptance decision is made.")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 7.3 — Refit frozen pipeline on all labelled rows
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/12_model_finalisation_and_packaging.md
# Workflow heading: Phase 12 > Core tasks checklist
# Checklist: Refit the selected preprocessing+estimator pipeline on approved final labelled data.
# Data scope: All and only labelled train rows; gated.
# Expected output / gate: final_model is fitted only after the one-time holdout decision is recorded.
# ============================================================================

final_model = None
X_all_labelled = None
y_all_labelled = None
if RUN_FINAL_SUBMISSION:
    assert holdout_metrics is not None
    assert holdout_spec_signature is not None
    assert HOLDOUT_DECISION_RECORDED
    assert stable_spec_signature(
        FINAL_MODEL_SPEC
    ) == holdout_spec_signature
    X_all_labelled = prepare_X_from_spec(
        labelled_data, FINAL_MODEL_SPEC
    )
    y_all_labelled = labelled_data[TARGET_COL]
    final_model = build_frozen_finalist()
    final_model.fit(X_all_labelled, y_all_labelled)
    print("Final pipeline refitted on labelled rows:", len(labelled_data))
else:
    print("Final refit skipped (safe default).")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 7.4 — Post-freeze external compatibility audit
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/11_final_evaluation.md
# - Machine Learning tổng/machine_learning_lifecycle_phases/12_model_finalisation_and_packaging.md
# Workflow heading: External exclusion from model choice / frozen input contract
# Checklist: Check schema, missingness, numeric support and unseen categories only after freeze.
# Data scope: Labelled support + external feature values; post-freeze and gated.
# Expected output / gate: Compatibility evidence may qualify limitations but must not trigger retuning.
# ============================================================================

external_compatibility_audit = None
X_external_frozen = None
if RUN_EXTERNAL_COMPATIBILITY_AUDIT:
    assert holdout_metrics is not None
    assert holdout_spec_signature is not None
    assert HOLDOUT_DECISION_RECORDED
    assert stable_spec_signature(
        FINAL_MODEL_SPEC
    ) == holdout_spec_signature
    X_labelled_frozen = prepare_X_from_spec(
        labelled_data, FINAL_MODEL_SPEC
    )
    X_external_frozen = prepare_X_from_spec(
        test_raw, FINAL_MODEL_SPEC
    )
    numeric_final_features = [
        col
        for col in FINAL_MODEL_SPEC["features"]
        if col != GROUP_COL
    ]
    external_rows = []
    for feature in numeric_final_features:
        labelled_min = X_labelled_frozen[feature].min()
        labelled_max = X_labelled_frozen[feature].max()
        external_rows.append(
            {
                "feature": feature,
                "labelled_min": labelled_min,
                "labelled_max": labelled_max,
                "external_min": X_external_frozen[feature].min(),
                "external_max": X_external_frozen[feature].max(),
                "external_missing": int(
                    X_external_frozen[feature].isna().sum()
                ),
                "below_labelled_min": int(
                    (
                        X_external_frozen[feature] < labelled_min
                    ).sum()
                ),
                "above_labelled_max": int(
                    (
                        X_external_frozen[feature] > labelled_max
                    ).sum()
                ),
            }
        )
    external_compatibility_audit = pd.DataFrame(external_rows)
    display(external_compatibility_audit)
    if GROUP_COL in FINAL_MODEL_SPEC["features"]:
        known_countries = set(X_labelled_frozen[GROUP_COL])
        unseen_rows = int(
            (~X_external_frozen[GROUP_COL].isin(known_countries)).sum()
        )
        print("Rows with unseen one-hot Country:", unseen_rows)
    print("Post-freeze audit only: do not retune from this evidence.")
else:
    print("External-value audit remains closed (safe default).")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 7.5 — Generate continuous external predictions
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/12_model_finalisation_and_packaging.md
# Workflow heading: Phase 12 > Confirm prediction generation and I/O contract
# Checklist: Predict in original external-row order without clipping/rounding unless required.
# Data scope: Frozen external features; gated.
# Expected output / gate: external_predictions has one finite candidate value per external row.
# ============================================================================

external_predictions = None
if RUN_FINAL_SUBMISSION:
    assert final_model is not None
    assert X_external_frozen is not None
    external_predictions = final_model.predict(X_external_frozen)
    assert len(external_predictions) == len(test_raw)
    print("Generated external predictions:", len(external_predictions))
else:
    print("External prediction generation skipped (safe default).")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 7.6 — Construct and validate submission contract
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/12_model_finalisation_and_packaging.md
# Workflow heading: Input/output contract and smoke tests
# Checklist: Validate header order, ID order, uniqueness, row count, missingness and finiteness.
# Data scope: External IDs + generated predictions; gated.
# Expected output / gate: submission matches exactly ID,TARGET_Capacity and the sample-template order.
# ============================================================================

def validate_submission(
    submission_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
) -> None:
    assert list(submission_frame.columns) == ["ID", TARGET_COL]
    assert len(submission_frame) == len(test_frame)
    assert (
        submission_frame["ID"].tolist()
        == test_frame[ID_COL].tolist()
    )
    assert submission_frame["ID"].is_unique
    assert submission_frame[TARGET_COL].notna().all()
    assert np.isfinite(submission_frame[TARGET_COL]).all()
    assert (
        submission_frame["ID"].tolist()
        == sample_raw["ID"].tolist()
    )

submission = None
if RUN_FINAL_SUBMISSION:
    assert external_predictions is not None
    assert (
        len(STUDENT_ID) == 8
        and STUDENT_ID.startswith("s")
        and STUDENT_ID[1:].isdigit()
    )
    submission = pd.DataFrame(
        {
            "ID": test_raw[ID_COL].to_numpy(),
            TARGET_COL: external_predictions,
        }
    )
    validate_submission(submission, test_raw)
    display(
        submission[TARGET_COL]
        .describe()
        .to_frame("external_prediction_audit")
    )
    display(submission.head())
else:
    print("Submission construction skipped (safe default).")


In [ ]:
# ============================================================================
# NOTEBOOK STEP 7.7 — Write and round-trip the exact prediction CSV
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/12_model_finalisation_and_packaging.md
# Workflow heading: Reproducible packaged output and smoke tests
# Checklist: Use exact filename; refuse overwrite; reread and revalidate values/order.
# Data scope: Validated in-memory submission; gated and writes one CSV.
# Expected output / gate: Saved file survives round-trip validation with unchanged predictions.
# ============================================================================

submission_path = None
if RUN_FINAL_SUBMISSION:
    assert submission is not None
    submission_path = (
        ASSIGNMENT_DIR
        / f"COSC2753_A1_Predictions_{STUDENT_ID}.csv"
    )
    assert not submission_path.exists(), (
        f"Refusing to overwrite existing file: {submission_path}"
    )
    submission.to_csv(submission_path, index=False)
    round_trip = pd.read_csv(submission_path)
    validate_submission(round_trip, test_raw)
    assert np.allclose(
        round_trip[TARGET_COL],
        submission[TARGET_COL],
    )
    print("Validated submission saved to:", submission_path)
else:
    print("CSV export skipped (safe default).")


### Phase 7 decision gate

- [ ] Holdout opened once after model/rule freeze.
- [ ] Result interpreted against thresholds written before access.
- [ ] Holdout failure, if any, is reported without repeated retuning.
- [ ] Final pipeline refitted on all and only labelled rows.
- [ ] CSV has exact headers/order/count/ID mapping/finite continuous predictions.

**Decision:** TODO — state whether the final evidence supports this offline
submission and what additional evidence real deployment would require.


# PHASE 8 — Report, Packaging & Reproducibility

**Lifecycle coverage:** `12_model_finalisation_and_packaging.md`

| Notebook step | Concrete purpose | Exact workflow file |
|---|---|---|
| 8.1 | Write executive summary | `12_model_finalisation_and_packaging.md` |
| 8.2 | Maintain report-ready evidence register | `12_model_finalisation_and_packaging.md` |
| 8.3 | Check cross-deliverable consistency | `12_model_finalisation_and_packaging.md` |
| 8.4 | Record input/environment/run manifests | `12_model_finalisation_and_packaging.md` |
| 8.5 | Record fresh Restart Kernel → Run All verification | `12_model_finalisation_and_packaging.md` |
| 8.6 | Audit code-ZIP contents | `12_model_finalisation_and_packaging.md` |
| 8.7 | Complete references and appendix | `12_model_finalisation_and_packaging.md` |

Every code cell below repeats its workflow source, data scope, and expected
output/gate in comments. Heavy training, holdout access, and file export remain
behind explicit flags.


## 8.1 Executive summary scaffold

- Problem and supported offline prediction setting: TODO
- Evaluation design: TODO
- Compared model families: TODO
- Selected model and decisive evidence: TODO
- Main limitation: TODO

## 8.2 Report-ready evidence register

| Report claim | Notebook evidence | Final value/version checked? |
|---|---|---|
| Group-aware stress test is justified | Steps 2.8, 3.1–3.2 | TODO |
| Selected model beats baselines | Steps 4.7, 5.8 | TODO |
| Model has acceptable stability | Steps 5.9, 7.1–7.2 | TODO |
| Main failure mode/limitation | Steps 6.3–6.8 | TODO |
| CSV is valid | Steps 7.6–7.7 | TODO |

## 8.3 Cross-deliverable consistency

PDF, video, notebook, README and CSV must use the same split description,
metrics, model names, hyperparameters, selected model, values, limitations and
filename. The notebook is the evidence store, not the five-page report.


In [ ]:
# ============================================================================
# NOTEBOOK STEP 8.4A — Input-data manifest
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/12_model_finalisation_and_packaging.md
# Workflow heading: Version data, artefacts and dependencies
# Checklist: Record relative paths, sizes and checksums for all required inputs.
# Data scope: Filesystem/input bytes only.
# Expected output / gate: All inputs exist and can be matched to the modelling run.
# ============================================================================

input_manifest = source_inventory.copy()
display(input_manifest)
assert input_manifest["exists"].all()


In [ ]:
# ============================================================================
# NOTEBOOK STEP 8.4B — Run/configuration manifest
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/12_model_finalisation_and_packaging.md
# Workflow heading: Record data/code/configuration/seeds/versions
# Checklist: Capture decisions and versions needed to explain/reproduce the final run.
# Data scope: In-memory configuration and prior step metadata only.
# Expected output / gate: JSON manifest reflects current policies, split, final spec and environment.
# ============================================================================

run_manifest = {
    "student_id": STUDENT_ID,
    "random_state": RANDOM_STATE,
    "primary_metric_key": PRIMARY_METRIC_KEY,
    "development_policy_confirmed": DEVELOPMENT_POLICY_CONFIRMED,
    "status_invalid_policy": STATUS_INVALID_POLICY,
    "country_feature_policy": COUNTRY_FEATURE_POLICY,
    "feature_columns": FEATURE_COLUMNS,
    "development_rows": len(development_df),
    "holdout_rows": len(holdout_df),
    "labelled_rows": len(labelled_data),
    "excluded_missing_target_rows": len(
        excluded_unlabelled_train_rows
    ),
    "group_cv_folds": N_GROUP_FOLDS,
    "final_spec_confirmed": FINAL_SPEC_CONFIRMED,
    "final_model_spec": FINAL_MODEL_SPEC,
    "frozen_final_spec_signature": frozen_final_spec_signature,
    "holdout_spec_signature": holdout_spec_signature,
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "sklearn_version": sklearn.__version__,
}
print(json.dumps(run_manifest, indent=2, default=str))


## 8.5 Fresh Restart Kernel → Run All verification

Do not attempt to restart the kernel from inside the notebook. Perform a fresh
external run, then record:

- date/environment: TODO
- flags enabled: TODO
- wall-clock runtime: TODO
- first failing cell, if any: TODO
- expected artefacts and hashes: TODO
- stale/manual output removed: TODO


In [ ]:
# ============================================================================
# NOTEBOOK STEP 8.5 — Execution-gate state for Run-All verification
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/12_model_finalisation_and_packaging.md
# Workflow heading: Smoke tests and reproduction
# Checklist: Make the exact enabled/disabled stages visible in the final run record.
# Data scope: Configuration only.
# Expected output / gate: Gate table prevents an ambiguous claim that a partial run was a full final run.
# ============================================================================

execution_gate_state = pd.Series(
    {
        "RUN_PREPROCESSING_SMOKE_TEST":
            RUN_PREPROCESSING_SMOKE_TEST,
        "RUN_POLICY_ABLATIONS": RUN_POLICY_ABLATIONS,
        "RUN_BASELINE_EVALUATION": RUN_BASELINE_EVALUATION,
        "RUN_MODEL_COMPARISON": RUN_MODEL_COMPARISON,
        "RUN_TUNING": RUN_TUNING,
        "RUN_OOF_ANALYSIS": RUN_OOF_ANALYSIS,
        "RUN_ROBUSTNESS_TESTS": RUN_ROBUSTNESS_TESTS,
        "RUN_FINALIST_ABLATIONS": RUN_FINALIST_ABLATIONS,
        "RUN_MODEL_INTERPRETATION": RUN_MODEL_INTERPRETATION,
        "RUN_FINAL_HOLDOUT": RUN_FINAL_HOLDOUT,
        "RUN_EXTERNAL_COMPATIBILITY_AUDIT":
            RUN_EXTERNAL_COMPATIBILITY_AUDIT,
        "RUN_FINAL_SUBMISSION": RUN_FINAL_SUBMISSION,
    },
    name="enabled",
)
display(execution_gate_state.to_frame())


In [ ]:
# ============================================================================
# NOTEBOOK STEP 8.6 — Read-only code-ZIP content audit
#
# WORKFLOW SOURCE
# - Machine Learning tổng/machine_learning_lifecycle_phases/12_model_finalisation_and_packaging.md
# Workflow heading: Artefact/version/checksum/approval checklist
# Checklist: Check expected source/config files and flag caches/checkpoints/stale outputs.
# Data scope: ASM1 directory names only; no archive is written.
# Expected output / gate: Missing expected items and forbidden package contents are visible before ZIP creation.
# ============================================================================

expected_code_items = [
    ASSIGNMENT_DIR / "ASM1_master_notebook.ipynb",
    ASSIGNMENT_DIR / "README.md",
    ASSIGNMENT_DIR / "requirements.txt",
]
code_package_audit = pd.DataFrame(
    {
        "relative_path": [
            str(path.relative_to(ASSIGNMENT_DIR))
            for path in expected_code_items
        ],
        "exists": [path.exists() for path in expected_code_items],
    }
)
forbidden_package_names = {
    "__pycache__",
    ".ipynb_checkpoints",
    ".DS_Store",
}
forbidden_found = [
    str(path.relative_to(ASSIGNMENT_DIR))
    for path in ASSIGNMENT_DIR.rglob("*")
    if path.name in forbidden_package_names
]
display(code_package_audit)
print("Forbidden/cache paths found:", forbidden_found)
print(
    "TODO: confirm the exact code-ZIP required contents from the current "
    "submission instruction before creating the archive."
)


## 8.7 References and appendix guidance

- Cite the assignment/submission PDFs actually used.
- Cite Weeks 1–5 lecture material for taught concepts.
- For a beyond-class estimator or diagnostic, cite an official/primary source
  actually consulted and explain why the method follows from EDA.
- Re-check near submission: report page limit, current video duration,
  official hidden-test metric, exact filenames and upload locations.

# Appendix A — Experiment register

| ID | Date | Data/split | Features | Pipeline/model | Search | Primary ± SD | MAE | R² | Gap | Runtime | Decision |
|---|---|---|---|---|---|---:|---:|---:|---:|---:|---|
| EXP-001 | TODO | seed 42 | TODO | baselines | none | TODO | TODO | TODO | TODO | TODO | TODO |
| EXP-002 | TODO | same folds | TODO | candidate | TODO | TODO | TODO | TODO | TODO | TODO | TODO |

# Appendix B — Reference register

- Assignment brief and submission-instruction PDFs in the ASM1 folder.
- `Machine Learning tổng/machine_learning_lifecycle_phases/README.md` and
  lifecycle files 01–12 used as the process standard.
- COSC2753 lecture slides, Weeks 1–5.
- TODO: official scikit-learn/primary references actually used.

# Final notebook gate

- [ ] Every required TODO is resolved.
- [ ] Every displayed number comes from the final reproducible run.
- [ ] Final claim follows from evidence and states limitations.
- [ ] Submission CSV passes all assertions.
- [ ] Report/video/code/CSV are aligned.
